# SC4000 v3 — graph coverage and training data in one run

This notebook contains **both experiments**:

1. Show the same GNN more of each graph, while keeping its training examples fixed.
2. At the development-selected graph budget, give it more training graphs and configurations.

It also tests masks/context, message passing, connected-neighbourhood sampling, matched-data
HGB controls, larger tabular training sets, a deeper HGB and Extra Trees. It chooses models
and blends automatically and exports full Kaggle candidate CSVs. You do not choose experiments
between stages. Everything needed is embedded here; no repository checkout is required.
The GNN coverage/data study runs on XLA layout default and random. NLP layout default/random
and tile receive the expanded tabular study. Submissions cover all five collections.

**Target:** standard Colab CPU RAM (approximately 12–13 GB), local data and temporary files.
No Google Drive mount is used. Checkpoint ZIPs are requested as downloads to your computer
every 30 minutes and after each collection, including while a long worker runs. The browser
must allow multiple downloads. **A download request is not proof that the ZIP reached your
computer. A disconnected browser cannot be relied on to receive new backups.** Check Downloads.

A nine-hour session budget does not mean the study finishes in nine hours. This is an extensive
staged study and can require more than one session. Resume the latest downloaded **v3** ZIP
with this same notebook/settings. Completed jobs are reused; GNNs resume at epoch checkpoints.
V2 results remain your historical benchmark and are not compatible v3 recovery archives.

The reported v2 Kaggle score was 0.45936, with a user-reported top-10% target around 0.59.
No improvement, completion time or ranking is guaranteed. The v3 baseline is retrained under
its own folds/sampling; it does not reproduce the exact v2 submission automatically.

## 1. Set paths and resource limits

For a fresh run, leave `RESTORE_ZIP` empty. It is an **input backup to restore**, not the output
path. Output is created automatically at `/content/sc4000_gnn_v3_results.zip` in Colab.
After a runtime reset, upload your most recent downloaded v3 ZIP and enter its actual path.
Use the same `RUN_NAME`. Local files from the deleted VM cannot be recovered without a copy.

Training batches stay at 16 configurations across graph budgets. We do not silently reduce
batch size, because that would change which pairs the ranking loss sees. A label-free pilot
rejects a graph budget if it exceeds the pilot resource limit; rejected budgets are logged.

In [ ]:
from pathlib import Path
import os, sys
DATA_ROOT = ""  # folder containing layout/ and tile/, or parent containing npz_all/npz
SAMPLE_SUBMISSION = ""  # autodetect beside competition data
DOWNLOAD_IF_MISSING = True
RUN_NAME = "sc4000_gnn_v3"
RESTORE_ZIP = ""  # e.g. "/content/sc4000_gnn_v3_results.zip"
AUTO_DOWNLOAD_CHECKPOINTS = True
CHECKPOINT_MINUTES = 30
SESSION_HOURS = 9.0
WORKER_RAM_GIB = 5.0
SYSTEM_RESERVE_GIB = 2.0
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_DIR / RUN_NAME
SCRATCH_DIR = BASE_DIR / ("." + RUN_NAME + "_scratch")
RUNTIME_DIR = BASE_DIR / ("." + RUN_NAME + "_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[name] = "2"
os.environ["MPLCONFIGDIR"] = str(RUNTIME_DIR / "matplotlib")

## 2. Dependencies and embedded implementation

Only missing dependencies are installed, without a pip download cache. A restored run requires
matching scientific-library versions; the saved identity prevents mixing incompatible results.
Only restore your own trusted archives: model artifacts include Python joblib files.

In [ ]:
import importlib.util, subprocess
packages = {"numpy":"numpy", "pandas":"pandas", "scipy":"scipy", "sklearn":"scikit-learn",
            "torch":"torch>=2.6", "joblib":"joblib", "psutil":"psutil",
            "threadpoolctl":"threadpoolctl", "matplotlib":"matplotlib"}
missing = [package for module,package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-cache-dir",*missing])

In [ ]:
# Embedded implementation: engine.py
SOURCE = r'''"""Independent, bounded-memory SC4000 experiments. Embedded in the Colab notebook.

Only original NPZ files and compact sampled feature matrices are read. No full
configuration array is decompressed into memory or an expanded disk cache.
"""
import argparse
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import sys
import tempfile
import time
import traceback
import zipfile

import joblib
import numpy as np
import pandas as pd
import psutil
from scipy.stats import kendalltau, rankdata
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

VERSION = '3.0.0'
GIB = 1024 ** 3
COLLECTIONS = ['layout:xla:default', 'layout:xla:random',
               'layout:nlp:default', 'layout:nlp:random', 'tile:xla']
BLOCKS = ['basic', 'masked', 'position', 'opcode', 'depth', 'edge', 'topology', 'wl']


def stable_seed(value):
    return int(hashlib.sha256(str(value).encode()).hexdigest()[:8], 16)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temp.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def release_memory():
    gc.collect()
    if sys.platform == 'linux':
        try:
            import ctypes
            ctypes.CDLL('libc.so.6').malloc_trim(0)
        except (OSError, AttributeError):
            pass


def guard(required=0, reserve=1.5 * GIB):
    if psutil.virtual_memory().available < required + reserve:
        raise MemoryError(f'Insufficient available RAM for {required / GIB:.2f} GiB allocation')


def header(stream):
    version = np.lib.format.read_magic(stream)
    if version == (1, 0):
        shape, order, dtype = np.lib.format.read_array_header_1_0(stream)
    elif version in ((2, 0), (3, 0)):
        shape, order, dtype = np.lib.format.read_array_header_2_0(stream)
    else:
        raise ValueError(f'Unsupported NPY version: {version}')
    if np.dtype(dtype).hasobject:
        raise ValueError('Object arrays are not allowed')
    return tuple(shape), order, np.dtype(dtype)


def metadata(path, key):
    with zipfile.ZipFile(path) as z, z.open(key + '.npy') as f:
        return header(f)


def exact_read(f, size):
    data = bytearray(size)
    view = memoryview(data)
    position = 0
    while position < size:
        count = f.readinto(view[position:])
        if not count:
            raise EOFError('Truncated NPZ member')
        position += count
    return data


def selected_rows(path, key, indices, chunk_bytes=16 * 1024 ** 2):
    """Yield one selected row at a time; discard skipped bytes in bounded chunks.

    Fortran arrays fail explicitly: silently loading their full payload is unsafe.
    """
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) and (np.any(np.diff(indices) <= 0) or indices[0] < 0):
        raise ValueError('Indices must be sorted, unique and nonnegative')
    with zipfile.ZipFile(path) as z, z.open(key + '.npy') as f:
        shape, order, dtype = header(f)
        if order:
            raise ValueError('Fortran-order configuration array: safe streaming requires C order')
        if len(indices) and indices[-1] >= shape[0]:
            raise IndexError('Configuration index out of range')
        size = math.prod(shape[1:]) * dtype.itemsize
        guard(size * 12)
        position = 0
        for index in indices:
            skip = (int(index) - position) * size
            while skip:
                chunk = f.read(min(skip, chunk_bytes))
                if not chunk:
                    raise EOFError('Truncated NPZ while skipping rows')
                skip -= len(chunk)
            raw = exact_read(f, size)
            row = np.frombuffer(raw, dtype=dtype).reshape(shape[1:])
            yield int(index), row
            del row, raw
            position = int(index) + 1


def read_small(path, key, limit=256 * 1024 ** 2):
    shape, _, dtype = metadata(path, key)
    size = math.prod(shape) * dtype.itemsize
    if size > limit:
        raise MemoryError(f'Static member {key} exceeds {limit / 1024**2:.0f} MiB guard')
    guard(size * 3)
    with np.load(path, allow_pickle=False) as data:
        return data[key]


def family(stem):
    for name in ['resnet', 'bert', 'albert', 'inception', 'efficientnet', 'transformer',
                 'retinanet', 'mask_rcnn', 'mnasnet', 'alexnet', 'magenta', 'brax',
                 'ncf', 'xception', 'electra', 'trax', 'unet', 'mlperf']:
        if name in stem.lower():
            return name
    if len(stem) >= 24 and all(c in '0123456789abcdef' for c in stem[:24]):
        return 'hashed_graph'
    return stem.split('_')[0].split('.')[0].split('-')[0]


def balanced_order(paths, seed):
    rng = np.random.default_rng(seed)
    groups = {}
    for path in sorted(paths):
        groups.setdefault(family(path.stem), []).append(path)
    for values in groups.values():
        rng.shuffle(values)
    names = sorted(groups)
    rng.shuffle(names)
    result = []
    while any(groups.values()):
        for name in names:
            if groups[name]:
                result.append(groups[name].pop())
    return result


def make_manifest(root, collection, cfg):
    folder = Path(root).joinpath(*collection.split(':'))
    train = sorted((folder / 'train').glob('*.npz'))
    valid = sorted((folder / 'valid').glob('*.npz'))
    if len(train) < 5 or not valid:
        raise ValueError(f'{collection}: need >=5 training graphs and >=1 official validation graph')
    if {p.stem for p in train} & {p.stem for p in valid}:
        raise ValueError('Graph names overlap between train and official validation')
    tile = collection.startswith('tile')
    order = balanced_order(train, cfg['split_seed'])
    ndev = min(cfg['dev_graphs'], max(2, len(order) // 5))
    dev = order[:ndev]
    fit = order[ndev:ndev + (cfg['tile_train_graphs'] if tile else cfg['layout_train_graphs'])]
    holdout = balanced_order(valid, cfg['split_seed'] + 1)[:cfg['holdout_graphs']]
    records = []
    for split, paths in [('train', fit), ('dev', dev), ('holdout', holdout)]:
        for path in sorted(paths):
            # ZIP CRCs bind resume to the source payload, independent of mount path/mtime.
            with zipfile.ZipFile(path) as z:
                source = [(i.filename, i.CRC, i.file_size) for i in z.infolist()]
            key = 'config_feat' if tile else 'node_config_feat'
            shape, order, dtype = metadata(path, key)
            n = shape[0]
            cap = cfg['train_pool_configs'] if split == 'train' else cfg['eval_configs']
            ids = np.sort(np.random.default_rng(stable_seed(f'{cfg["split_seed"]}/{collection}/{path.stem}'))
                          .choice(n, min(n, cap), replace=False))
            records.append(dict(split=split, graph=path.stem, family=family(path.stem),
                                relative_path=str(path.relative_to(root)), count=n,
                                indices=ids.tolist(), source=digest(source),
                                fortran=bool(order), row_bytes=math.prod(shape[1:]) * dtype.itemsize))
    return dict(collection=collection, records=records,
                split_note='dev graphs from official train; holdout from official valid; no test labels')


def stats(values):
    a = np.asarray(values, dtype=np.float64)
    if not a.size:
        return np.zeros(4, dtype=np.float32)
    return np.array([a.mean(), a.std(), a.min(), a.max()], dtype=np.float32)


def hash_counts(values, bins):
    return np.bincount(np.asarray(values, dtype=np.uint64).ravel().astype(np.int64) % bins,
                       minlength=bins).astype(np.float32)


def graph_context(path, cfg):
    op = read_small(path, 'node_opcode').astype(np.int64)
    feat = read_small(path, 'node_feat').astype(np.float32)
    edge = read_small(path, 'edge_index').astype(np.int64)
    n = len(op)
    if not n or edge.ndim != 2 or edge.shape[1] != 2 or feat.shape[0] != n:
        raise ValueError('Invalid graph arrays')
    if edge.size and (edge.min() < 0 or edge.max() >= n):
        raise ValueError('Edge node index out of range')
    # Official NPZ convention: [consumer, producer], opposite to common edge lists.
    consumer, producer = edge[:, 0], edge[:, 1]
    indegree = np.bincount(consumer, minlength=n)
    outdegree = np.bincount(producer, minlength=n)
    # Kahn traversal using sorted CSR adjacency, avoiding Python lists per edge.
    order = np.argsort(producer, kind='stable')
    targets = consumer[order]
    offsets = np.concatenate([[0], np.cumsum(outdegree)])
    left = indegree.copy()
    queue = np.empty(n, dtype=np.int64)
    roots = np.flatnonzero(left == 0)
    queue[:len(roots)] = roots
    tail, head = len(roots), 0
    depth = np.zeros(n, dtype=np.int64)
    while head < tail:
        u = queue[head]
        head += 1
        for v in targets[offsets[u]:offsets[u + 1]]:
            depth[v] = max(depth[v], depth[u] + 1)
            left[v] -= 1
            if left[v] == 0:
                queue[tail] = v
                tail += 1
    feat = np.nan_to_num(feat, nan=0, posinf=0, neginf=0)
    signed = np.sign(feat) * np.log1p(np.abs(feat))
    # Fixed width preserves compatibility across graphs; upstream schema is checked per collection.
    basic = np.concatenate([np.log1p([n, len(edge)]), signed.mean(0), signed.std(0),
                            signed.min(0), signed.max(0), hash_counts(op, 32) / n]).astype(np.float32)
    topology = np.concatenate([stats(np.log1p(indegree)), stats(np.log1p(outdegree)),
                               stats(np.log1p(depth)), [np.mean(left > 0)]]).astype(np.float32)
    labels = op.copy() % 64
    wl_parts = []
    for _ in range(2):
        incoming = np.bincount(consumer, weights=(labels[producer] + 1) * 37,
                               minlength=n).astype(np.int64)
        outgoing = np.bincount(producer, weights=(labels[consumer] + 1) * 53,
                               minlength=n).astype(np.int64)
        labels = (labels * 1009 + incoming * 9176 + outgoing * 6361) % 64
        wl_parts.append(hash_counts(labels, 64) / n)
    # This bounded hashed neighborhood signature is our implementation, not exact WL isomorphism testing.
    return dict(op=op, numeric=signed, edge=edge, depth=depth, basic=basic, topology=topology,
                wl=np.concatenate(wl_parts), nodes=n, cycle_fraction=float(np.mean(left > 0)))


def layout_context(path, context, cfg):
    ids = read_small(path, 'node_config_ids').astype(np.int64)
    n = context['nodes']
    if not len(ids) or ids.min() < 0 or ids.max() >= n or len(np.unique(ids)) != len(ids):
        raise ValueError('Invalid configurable node IDs')
    lookup = np.full(n, -1, dtype=np.int64)
    lookup[ids] = np.arange(len(ids))
    pairs = lookup[context['edge']]
    pairs = pairs[np.all(pairs >= 0, axis=1)]
    # Use a deterministic, configuration-independent edge cap.
    if len(pairs) > cfg['edge_cap']:
        chosen = np.sort(np.random.default_rng(913).choice(len(pairs), cfg['edge_cap'], replace=False))
        pairs = pairs[chosen]
    d = context['depth'][ids]
    bins = np.minimum(3, (d * 4 // max(1, int(context['depth'].max()) + 1)))
    return dict(ids=ids, opcode=context['op'][ids], depth=bins, pairs=pairs)


def layout_features(row, context, local, cfg):
    a = np.asarray(row)
    if a.shape != (len(local['ids']), 18):
        raise ValueError(f'Expected layout shape (configurable_nodes,18), got {a.shape}')
    if not np.isfinite(a).all() or np.any(a != np.floor(a)) or a.min() < -1 or a.max() > 5:
        raise ValueError('Layout values must be categorical indices -1..5')
    mask = a != -1
    zero = np.where(mask, a, 0).astype(np.float32)
    basic_config = np.concatenate([zero.mean(0), zero.std(0), zero.min(0), zero.max(0),
                                   [1 - mask.mean(), np.log1p(len(a))]])
    count = mask.sum(0)
    means = zero.sum(0) / np.maximum(1, count)
    variance = (zero ** 2).sum(0) / np.maximum(1, count) - means ** 2
    minimum = np.where(mask, a, 6).min(0)
    maximum = np.where(mask, a, -1).max(0)
    minimum[count == 0], maximum[count == 0] = 0, 0
    masked = np.concatenate([means, np.sqrt(np.maximum(0, variance)), minimum, maximum,
                             count / len(a)]).astype(np.float32)
    position = np.zeros((18, 7), dtype=np.float32)
    opcode_hist = np.zeros(cfg['opcode_bins'], dtype=np.float32)
    depth_hist = np.zeros(cfg['depth_bins'], dtype=np.float32)
    # Include padding as its own category; never treat axis ID as a physical magnitude here.
    categories = a.astype(np.int64) + 1
    for p in range(18):
        token = categories[:, p]
        position[p] = np.bincount(token, minlength=7) / len(a)
        opcode_hist += hash_counts(local['opcode'] * 1009 + p * 37 + token * 9176, cfg['opcode_bins'])
        depth_hist += hash_counts(local['depth'] * 1009 + p * 37 + token * 9176, cfg['depth_bins'])
    denom = max(1, len(a) * 18)
    pairs = local['pairs']
    edge_hist = np.zeros(cfg['edge_bins'], dtype=np.float32)
    if len(pairs):
        # Categorical co-occurrence, not an assertion of physical layout compatibility.
        for p in [0, 6, 12]:
            codes = categories[pairs[:, 0], p] * 7 + categories[pairs[:, 1], p]
            edge_hist += hash_counts(codes + p * 1009, cfg['edge_bins'])
        edge_hist /= len(pairs) * 3
    return dict(basic=np.concatenate([context['basic'], basic_config]).astype(np.float32),
                masked=masked, position=position.ravel(), opcode=opcode_hist / denom,
                depth=depth_hist / denom, edge=edge_hist,
                topology=context['topology'], wl=context['wl'])


def tile_features(row, context, cfg):
    a = np.asarray(row, dtype=np.float32).ravel()
    if len(a) != 24:
        raise ValueError(f'Expected 24 tile features, got {len(a)}')
    a = np.nan_to_num(a, nan=0, posinf=0, neginf=0)
    signed = np.sign(a) * np.log1p(np.abs(a))
    # Tile-specific transformed values and interactions; layout categories do not apply.
    interactions = np.outer(signed[:8], signed[:8])[np.triu_indices(8, 1)]
    return dict(basic=np.concatenate([context['basic'], a, stats(a)]).astype(np.float32),
                masked=signed, position=interactions.astype(np.float32),
                opcode=np.zeros(1, np.float32), depth=np.zeros(1, np.float32),
                edge=np.zeros(1, np.float32), topology=context['topology'], wl=context['wl'])


def extract_graph(root, record, collection, cfg, target):
    path = Path(root) / record['relative_path']
    context = graph_context(path, cfg)
    tile = collection.startswith('tile')
    local = None if tile else layout_context(path, context, cfg)
    gnn_graph = None
    if collection in cfg.get('gnn_collections', []):
        from study_gnn import compact_graph
        gnn_graph = compact_graph(context, local, cfg)
    key = 'config_feat' if tile else 'node_config_feat'
    runtime = read_small(path, 'config_runtime').astype(np.float64)
    if tile:
        normalizer = read_small(path, 'config_runtime_normalizers').astype(np.float64)
        if np.any(normalizer <= 0) or len(normalizer) != len(runtime):
            raise ValueError('Invalid tile runtime normalizers')
        runtime /= normalizer
    ids = np.asarray(record['indices'], dtype=np.int64)
    if len(runtime) != record['count'] or not np.isfinite(runtime[ids]).all() or np.any(runtime[ids] <= 0):
        raise ValueError('Invalid runtime labels')
    arrays, fingerprints = None, []
    layouts = None
    if gnn_graph is not None:
        size = len(ids)*len(gnn_graph['gnn_positions'])*18
        guard(size*3)
        layouts = np.empty((len(ids),len(gnn_graph['gnn_positions']),18),np.int8)
    for j, (index, row) in enumerate(selected_rows(path, key, ids)):
        features = tile_features(row, context, cfg) if tile else layout_features(row, context, local, cfg)
        if arrays is None:
            required = len(ids) * sum(v.size for v in features.values()) * 4
            if required > cfg['max_graph_features_mib'] * 1024**2:
                raise MemoryError('Compact graph feature matrix exceeds configured limit')
            guard(required * 3)
            arrays = {k: np.empty((len(ids), len(v)), np.float32) for k, v in features.items()}
        for name, values in features.items():
            if not np.isfinite(values).all():
                raise ValueError('Non-finite features')
            arrays[name][j] = values
        fingerprints.append(hashlib.blake2b(np.ascontiguousarray(row).tobytes(), digest_size=16).hexdigest())
        if layouts is not None:
            layouts[j] = row[gnn_graph['gnn_positions']].astype(np.int8)
    if arrays is None:
        raise ValueError('No configurations selected')
    if gnn_graph is not None:
        arrays.update(gnn_graph)
        arrays['gnn_layout'] = layouts
    np.savez_compressed(target, **arrays, y=runtime[ids], ids=ids,
                        fingerprints=np.array(fingerprints, dtype='U32'))
    return dict(sampled_configs=len(ids), unique_configs=len(set(fingerprints)),
                cycle_fraction=context['cycle_fraction'],
                gnn_coverage=None if gnn_graph is None else gnn_graph['gnn_coverage'].tolist(),
                gnn_isolated_fraction=None if gnn_graph is None else float(gnn_graph['gnn_isolated_fraction'][0]))


def prepare(root, manifest, cfg, scratch, output):
    """Compact artifacts only, with checkpoints per graph and no raw-array cache."""
    scratch, output = Path(scratch), Path(output)
    scratch.mkdir(parents=True, exist_ok=True)
    failures, prepared = [], []
    for k, record in enumerate(manifest['records']):
        target = scratch / f'g{k:05d}.npz'
        status = scratch / f'g{k:05d}.json'
        signature = digest(dict(record=record, version=VERSION, features={
            key: cfg.get(key) for key in ['opcode_bins','depth_bins','edge_bins','edge_cap','gnn_collections','gnn_nodes','gnn_config_nodes','gnn_edges','gnn_sampler']}))
        try:
            if target.exists() and status.exists() and json.loads(status.read_text()).get('signature') == signature:
                details = json.loads(status.read_text())['details']
            else:
                print(f'Extract {k+1}/{len(manifest["records"])} {record["split"]} {record["graph"]}', flush=True)
                details = extract_graph(root, record, manifest['collection'], cfg, target)
                write_json(status, dict(signature=signature, details=details))
            prepared.append(dict(record, artifact=target.name, **details))
        except Exception as exc:
            target.unlink(missing_ok=True)
            failures.append(dict(graph=record['graph'], split=record['split'], error=str(exc)))
            print(f'Graph excluded for every experiment: {record["graph"]}: {exc}', flush=True)
        release_memory()
        if sum(p.stat().st_size for p in scratch.glob('*.npz')) > cfg['scratch_limit_gib'] * GIB:
            raise MemoryError('Compact scratch feature disk limit reached')
    write_json(output / 'extraction_failures.json', failures)
    write_json(output / 'coverage.json', prepared)
    write_json(scratch / 'prepared.json', prepared)
    counts = {s: sum(r['split'] == s for r in prepared) for s in ['train', 'dev', 'holdout']}
    if counts['train'] < 3 or counts['dev'] < 2 or counts['holdout'] < 1:
        raise ValueError(f'Insufficient shared coverage after extraction: {counts}')
    if len(failures) / max(1, len(manifest['records'])) > cfg['max_excluded_fraction']:
        raise ValueError('Too many excluded graphs; inspect extraction_failures.json')
    write_json(scratch / 'ready.json', dict(counts=counts, complete=True))


def experiments(tile=False):
    def trial(name, blocks, **kwargs):
        return dict(name=name, blocks=blocks, model='hgb', target='centered_log',
                    weighting='graph', dedup=False, row_fraction=1., graph_fraction=1.,
                    **kwargs) if not kwargs else dict(dict(name=name, blocks=blocks, model='hgb',
                    target='centered_log', weighting='graph', dedup=False,
                    row_fraction=1., graph_fraction=1.), **kwargs)
    base = ['basic']
    full = ['basic', 'masked', 'position', 'topology', 'wl'] if tile else [
        'basic', 'masked', 'position', 'opcode', 'depth', 'edge']
    result = [trial('B0_basic_hgb', base), trial('B1_masked_hgb', ['basic', 'masked']),
              trial('P0_position', ['basic', 'masked', 'position']), trial('P_full', full)]
    if not tile:
        result += [trial('P_opcode', ['basic', 'masked', 'position', 'opcode']),
                   trial('P_depth', ['basic', 'masked', 'position', 'depth']),
                   trial('P_opcode_depth', ['basic', 'masked', 'position', 'opcode', 'depth']),
                   trial('P_edge', ['basic', 'masked', 'position', 'edge'])]
    result += [trial('S_topology', base + ['topology']), trial('S_wl', base + ['wl']),
               trial('P_full_static', full + [b for b in ['topology', 'wl'] if b not in full])]
    if tile:
        # Avoid a duplicate trial: P_full already contains all tile feature blocks.
        result = [r for r in result if r['name'] != 'P_full_static']
    for block in full:
        if block != 'basic':
            result.append(trial('A_without_' + block, [b for b in full if b != block]))
    result += [trial('C_unweighted', full, weighting='none'),
               trial('C_family_weight', full, weighting='family'),
               trial('C_log_target', full, target='log'),
               trial('C_percentile_target', full, target='percentile'),
               trial('C_deduplicate', full, dedup=True),
               trial('C_half_configs', full, row_fraction=.5),
               trial('C_half_graphs', full, graph_fraction=.5),
               trial('M_extra_trees', full, model='extra_trees'),
               trial('M_mlp', full, model='mlp'),
               trial('M_linear_regression', full, model='ridge'),
               trial('M_linear_pairwise', full, model='pairwise')]
    return result


def feature_matrix(data, blocks):
    return np.concatenate([data[b] for b in blocks], axis=1).astype(np.float32, copy=False)


def training_records(records, exp, seed, cfg):
    records = [r for r in records if r['split'] == 'train']
    if exp.get('matched'):
        records = sorted(records,key=lambda r: stable_seed(f'gnn-train/{r["graph"]}'))[:cfg['gnn_train_graphs']]
    cap=exp.get('train_graphs',0)
    if cap:
        records=sorted(records,key=lambda r:stable_seed(f'gnn-train/{r["graph"]}'))[:cap]
    if exp['graph_fraction'] < 1:
        order = sorted(records, key=lambda r: stable_seed(f'{seed}/{r["graph"]}'))
        records = order[:max(2, math.ceil(len(order) * exp['graph_fraction']))]
    return records


def training_indices(n, graph, seed, cfg, matched=False, fraction=1.):
    cap = cfg['gnn_configs'] if matched else cfg['fit_configs']
    take = min(n,max(1,int(cap*fraction)))
    return np.sort(np.random.default_rng(stable_seed(f'{seed}/{graph}')).permutation(n)[:take])


def training_data(scratch, records, exp, seed, cfg):
    records = training_records(records,exp,seed,cfg)
    xs, ys, weights, groups = [], [], [], []
    family_counts = {}
    for r in records:
        family_counts[r['family']] = family_counts.get(r['family'], 0) + 1
    total_bytes = 0
    for group, record in enumerate(records):
        with np.load(Path(scratch) / record['artifact'], allow_pickle=False) as data:
            n = len(data['y'])
            selected = training_indices(n,record['graph'],seed,cfg,exp.get('matched',False),exp['row_fraction'])
            x = feature_matrix(data, exp['blocks'])[selected]
            y = data['y'][selected].astype(np.float64)
            if exp['dedup']:
                _, inverse = np.unique(data['fingerprints'][selected], return_inverse=True)
                unique_ids = np.unique(inverse)
                x = np.stack([x[np.flatnonzero(inverse == j)[0]] for j in unique_ids])
                y = np.array([np.median(y[inverse == j]) for j in unique_ids])
            if exp['target'] == 'percentile':
                y = (rankdata(y) - 1) / max(1, len(y) - 1)
            else:
                y = np.log1p(y)
                if exp['target'] == 'centered_log':
                    y -= np.median(y)
            w = np.ones(len(y), np.float64)
            if exp['weighting'] in ['graph', 'family']:
                w /= len(y)
            if exp['weighting'] == 'family':
                w /= family_counts[record['family']]
            total_bytes += x.nbytes
            if total_bytes > cfg['max_matrix_mib'] * 1024**2:
                raise MemoryError('Training matrix exceeds configured bound')
            guard(total_bytes * 5)
            xs.append(x); ys.append(y); weights.append(w)
            groups.append(np.full(len(y), group, np.int32))
    x, y, w, g = np.concatenate(xs), np.concatenate(ys), np.concatenate(weights), np.concatenate(groups)
    return x, y, w / w.mean(), g


def fit_model(x, y, weights, groups, exp, seed, cfg):
    kind = exp['model']
    scaler = None
    if kind in ['ridge', 'pairwise', 'mlp']:
        scaler = StandardScaler()
        x = scaler.fit_transform(x).astype(np.float32)
    if kind == 'hgb':
        model = HistGradientBoostingRegressor(max_iter=exp.get('iterations',cfg['hgb_iterations']), learning_rate=exp.get('learning_rate',.06),
                    max_leaf_nodes=exp.get('leaves',31), min_samples_leaf=exp.get('min_leaf',20), l2_regularization=.1,
                    early_stopping=False, random_state=seed)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'extra_trees':
        model = ExtraTreesRegressor(n_estimators=cfg['extra_trees'], max_depth=14,
                                     min_samples_leaf=8, n_jobs=cfg['cpu_threads'], random_state=seed)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'ridge':
        model = Ridge(alpha=10.)
        model.fit(x, y, sample_weight=weights)
    elif kind == 'mlp':
        model = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=cfg['mlp_epochs'],
                             batch_size=256, early_stopping=False, random_state=seed)
        # sklearn versions differ in weight support. Weighted resampling is explicit and stable.
        rng = np.random.default_rng(seed)
        rows = rng.choice(len(y), len(y), replace=True, p=weights / weights.sum())
        model.fit(x[rows], y[rows])
    elif kind == 'pairwise':
        # A separate linear comparator, not the existing GNN or a novelty claim.
        rng = np.random.default_rng(seed)
        left, right, pair_weights = [], [], []
        for group in np.unique(groups):
            ids = np.flatnonzero(groups == group)
            i = rng.choice(ids, cfg['pairs_per_graph'])
            j = rng.choice(ids, cfg['pairs_per_graph'])
            use = y[i] != y[j]
            i, j = i[use], j[use]
            left.extend(i); right.extend(j)
            pair_weights.extend(np.repeat(weights[ids].mean(), len(i)))
        left, right = np.asarray(left, dtype=int), np.asarray(right, dtype=int)
        if not len(left):
            raise ValueError('No unequal-runtime training pairs')
        difference = x[left] - x[right]
        labels = y[left] > y[right]
        # Mirrored pairs guarantee both classes and force antisymmetric supervision.
        difference = np.concatenate([difference, -difference])
        labels = np.concatenate([labels, ~labels])
        pw = np.tile(pair_weights, 2).astype(float)
        model = SGDClassifier(loss='log_loss', alpha=.001, max_iter=300, tol=1e-4,
                               fit_intercept=False, random_state=seed)
        model.fit(difference, labels, sample_weight=pw / pw.mean())
    else:
        raise ValueError(kind)
    return dict(model=model, scaler=scaler, experiment=exp)


def predict(bundle, x):
    if bundle['scaler'] is not None:
        x = bundle['scaler'].transform(x)
    if bundle['experiment']['model'] == 'pairwise':
        return bundle['model'].decision_function(x)
    return bundle['model'].predict(x)


def scores(y, pred, tile=False):
    """Exact tau-b on the sampled set; never call these Kaggle/public LB scores."""
    if not np.isfinite(pred).all():
        raise ValueError('Non-finite predictions')
    tau = kendalltau(y, pred, variant='b').statistic
    tau = 0. if not np.isfinite(tau) else float(tau)
    order = np.argsort(pred, kind='stable')
    regret = float(y[order[:5]].min() / y.min() - 1)
    return dict(score=1 - regret if tile else tau, kendall_tau_b=tau,
                top5_regret=regret, distinct_predictions=int(len(np.unique(pred))))


def evaluate(bundle, scratch, records, split, collection, output, seed, exp_name):
    rows, prediction_rows = [], []
    for record in records:
        if record['split'] != split:
            continue
        with np.load(Path(scratch) / record['artifact'], allow_pickle=False) as data:
            x = feature_matrix(data, bundle['experiment']['blocks'])
            y, ids = data['y'], data['ids']
            pred = predict(bundle, x)
            metric = scores(y, pred, collection.startswith('tile'))
            rows.append(dict(collection=collection, split=split, graph=record['graph'],
                             family=record['family'], experiment=exp_name, seed=seed,
                             configurations=len(y), total_configurations=record['count'], **metric))
            prediction_rows.append(pd.DataFrame(dict(graph=record['graph'], config_id=ids,
                                                      runtime=y, prediction=pred)))
    if not rows:
        raise ValueError(f'No {split} graphs')
    pd.DataFrame(rows).to_csv(Path(output) / f'{split}_graphs.csv', index=False)
    pd.concat(prediction_rows, ignore_index=True).to_csv(Path(output) / f'{split}_predictions.csv.gz',
                                                      index=False, compression='gzip')
    return float(np.mean([r['score'] for r in rows]))


def run_trial(job):
    if job['experiment']['model'] == 'gnn':
        from study_gnn import train
        return train(job)
    output = Path(job['output']); output.mkdir(parents=True, exist_ok=True)
    cfg, exp, seed = job['cfg'], job['experiment'], job['seed']
    records = json.loads((Path(job['scratch']) / 'prepared.json').read_text())
    start = time.monotonic()
    with threadpool_limits(limits=cfg['cpu_threads']):
        selected_records=training_records(records,exp,seed,cfg)
        train_ids=[dict(graph=r['graph'],config_ids=np.asarray(r['indices'])[
            training_indices(len(r['indices']),r['graph'],seed,cfg,exp.get('matched',False),exp['row_fraction'])].tolist()) for r in selected_records]
        write_json(output/'training_manifest.json',train_ids)
        x, y, w, groups = training_data(job['scratch'], records, exp, seed, cfg)
        fit_start = time.monotonic()
        bundle = fit_model(x, y, w, groups, exp, seed, cfg)
        fit_seconds = time.monotonic() - fit_start
        train_rows, feature_count = x.shape
        del x, y, w, groups
        release_memory()
        dev = evaluate(bundle, job['scratch'], records, 'dev', job['collection'], output, seed, exp['name'])
        joblib.dump(bundle, output / 'model.joblib', compress=3)
    write_json(output / 'result.json', dict(status='ok', experiment=exp['name'], seed=seed,
               collection=job['collection'], dev_score=dev, train_rows=train_rows,
               feature_count=feature_count, fit_seconds=fit_seconds,
               total_seconds=time.monotonic()-start, version=VERSION))


def evaluate_holdout(job):
    if job['experiment']['model'] == 'gnn':
        from study_gnn import holdout
        return holdout(job)
    bundle = joblib.load(Path(job['output']) / 'model.joblib')
    records = json.loads((Path(job['scratch']) / 'prepared.json').read_text())
    with threadpool_limits(limits=job['cfg']['cpu_threads']):
        score = evaluate(bundle, job['scratch'], records, 'holdout', job['collection'],
                         job['output'], job['seed'], job['experiment']['name'])
    write_json(Path(job['output']) / 'holdout_result.json', dict(status='ok', score=score))


def worker(job_path):
    job = json.loads(Path(job_path).read_text())
    try:
        if job['action'] == 'prepare':
            prepare(job['root'], job['manifest'], job['cfg'], job['scratch'], job['output'])
        elif job['action'] == 'trial':
            run_trial(job)
        elif job['action'] == 'holdout':
            evaluate_holdout(job)
        elif job['action'] == 'probe':
            from study_gnn import probe
            probe(job)
        elif job['action'] == 'inference':
            from study_submission import infer_graph
            infer_graph(job)
        else:
            raise ValueError(job['action'])
    except Exception:
        traceback.print_exc()
        return 1
    return 0


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    sys.exit(worker(args.job))
'''
(RUNTIME_DIR / "study_engine.py").write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded implementation: gnn.py
SOURCE = r'''"""CPU GraphSAGE reference, controlled additions, and bounded graph preparation.

Retains the existing project's two-layer, width-32 GraphSAGE approach. Strict
sampling, corrected producer-to-consumer edges, masks and shared manifests mean
this is a controlled reimplementation, not an exact historical reproduction.
"""
import copy
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from study_engine import (read_small, stable_seed, scores, write_json, release_memory,
                          training_records, training_indices)

GNN_COLLECTIONS = ['layout:xla:default', 'layout:xla:random']


def catalog():
    reference = dict(model='gnn', blocks=[], layers=2, width=32, masks=False,
                     context=False, message=True, opcode=True, numeric=True,
                     loss='pairwise', matched=True)
    def exp(name, **changes):
        return dict(reference, name=name, **changes)
    return [exp('G0_reference'), exp('G_add_masks', masks=True),
            exp('G_add_context', context=True), exp('G_full', masks=True, context=True),
            exp('G_no_messages', masks=True, context=True, message=False),
            exp('G_no_opcode_embedding', masks=True, context=True, opcode=False),
            exp('G_no_numeric', masks=True, context=True, numeric=False),
            exp('G_regression', masks=True, context=True, loss='regression'),
            exp('G_one_layer', masks=True, context=True, layers=1)]


def compact_graph(context, local, cfg):
    """Strict node AND edge bounds, before PyTorch allocation; no adjacency lists."""
    ids = local['ids']
    rng = np.random.default_rng(193)
    count = min(len(ids), cfg['gnn_config_nodes'], cfg['gnn_nodes'])
    # Fixed permutations give nested configurable seeds when only the budget increases.
    positions = np.sort(np.random.default_rng(193).permutation(len(ids))[:count])
    if cfg.get('gnn_sampler','uniform') == 'connected':
        # Separate ablation: choose seeds by graph distance to a fixed anchor.
        edges=context['edge'];n=context['nodes']
        distance=np.full(n,n+1,np.int64);distance[ids[positions[0]]]=0
        for hop in range(min(n,cfg['gnn_nodes'])):
            frontier=distance==hop
            touched=edges[np.any(frontier[edges],axis=1)].ravel()
            if not len(touched):break
            new=np.unique(touched[distance[touched]>hop+1]);distance[new]=hop+1
            if np.sum(distance[ids]<=hop+1)>=count:break
        tie=np.random.default_rng(193).permutation(len(ids))
        positions=np.sort(tie[np.argsort(distance[ids[tie]],kind='stable')[:count]])
    seeds = ids[positions]
    keep = np.sort(seeds)
    edges = context['edge']  # [consumer, producer]
    for _ in range(2):
        if len(keep) >= cfg['gnn_nodes']:
            break
        mask = np.zeros(context['nodes'], dtype=bool)
        mask[keep] = True
        candidates = np.unique(edges[np.any(mask[edges], axis=1)].ravel())
        candidates = candidates[~mask[candidates]]
        budget = min(len(candidates), cfg['gnn_nodes']-len(keep))
        chosen = rng.choice(candidates, budget, replace=False)
        keep = np.sort(np.concatenate([keep,chosen]))
    lookup = np.full(context['nodes'], -1, np.int64)
    lookup[keep] = np.arange(len(keep))
    mapped = lookup[edges]
    mapped = mapped[np.all(mapped >= 0,axis=1)]
    before = len(mapped)
    if before > cfg['gnn_edges']:
        chosen = np.sort(rng.choice(before,cfg['gnn_edges'],replace=False))
        mapped = mapped[chosen]
    numeric = context['numeric'][keep].copy()
    numeric = (numeric-numeric.mean(0))/(numeric.std(0)+1e-6)
    config_nodes = lookup[seeds]
    degree=np.bincount(mapped.ravel(),minlength=len(keep)) if len(mapped) else np.zeros(len(keep),int)
    isolated=float(np.mean(degree==0))
    assert len(keep) <= cfg['gnn_nodes'] and len(mapped) <= cfg['gnn_edges']
    return dict(gnn_isolated_fraction=np.array([isolated],np.float32),gnn_numeric=numeric.astype(np.float32), gnn_opcode=context['op'][keep].clip(0,255),
                gnn_src=mapped[:,1], gnn_dst=mapped[:,0],
                gnn_config_nodes=config_nodes, gnn_positions=positions,
                gnn_coverage=np.array([context['nodes'],len(keep),len(ids),len(positions),
                                       len(edges),before,len(mapped)],dtype=np.int64))


class Ranker(nn.Module):
    def __init__(self, numeric_dim, summary_dim, exp):
        super().__init__()
        self.exp = exp
        h = exp['width']
        self.embedding = nn.Embedding(256,16)
        # Same capacity for reference/mask/context additions: unused inputs are zeros.
        self.project = nn.Linear(numeric_dim+16+18+18,h)
        self.layers = nn.ModuleList([nn.Linear(2*h,h) for _ in range(exp['layers'])])
        self.head = nn.Sequential(nn.Linear(h*4+summary_dim,h),nn.ReLU(),nn.Dropout(.05),nn.Linear(h,1))

    def forward(self, graph, layout, summary):
        numeric, opcode, src, dst, config_nodes = graph
        batch, n = len(layout),len(numeric)
        features = numeric if self.exp['numeric'] else torch.zeros_like(numeric)
        op = self.embedding(opcode)
        if not self.exp['opcode']:
            op = torch.zeros_like(op)
        values = torch.zeros((batch,n,18),dtype=torch.float32)
        masks = torch.zeros_like(values)
        valid = layout != -1
        values[:,config_nodes] = torch.where(valid,layout,0).float()/5.
        if self.exp['masks']:
            masks[:,config_nodes] = valid.float()
        h = F.relu(self.project(torch.cat([features.unsqueeze(0).expand(batch,-1,-1),
                    op.unsqueeze(0).expand(batch,-1,-1),values,masks],dim=-1)))
        degree = torch.bincount(dst,minlength=n).clamp_min(1).float()
        for layer in self.layers:
            neighbors = torch.zeros_like(h)
            if self.exp['message'] and len(src):
                neighbors.index_add_(1,dst,h[:,src])
                neighbors /= degree[None,:,None]
            h = h + F.relu(layer(torch.cat([h,neighbors],dim=-1)))
        selected = h[:,config_nodes]
        pooled = torch.cat([h.mean(1),h.max(1).values,selected.mean(1),selected.max(1).values],dim=-1)
        if not self.exp['context']:
            summary = torch.zeros_like(summary)
        return self.head(torch.cat([pooled,summary],dim=-1)).flatten()


def arrays(data):
    graph = tuple(torch.from_numpy(np.asarray(data[k]).copy()) for k in
                  ['gnn_numeric','gnn_opcode','gnn_src','gnn_dst','gnn_config_nodes'])
    # Histograms remain small, finite and label independent.
    summary = np.concatenate([data[k] for k in ['position','opcode','depth','edge']],axis=1).astype(np.float32)
    return graph, data['gnn_layout'].copy(),summary


def loss_function(pred,y,mode):
    if mode == 'regression':
        return F.smooth_l1_loss(pred,y)
    left,right = torch.triu_indices(len(pred),len(pred),offset=1)
    difference = y[left]-y[right]
    eligible = difference.abs() > .02
    if not eligible.any():
        return F.smooth_l1_loss(pred,y)
    return F.relu(.1 - difference[eligible].sign() * (pred[left[eligible]]-pred[right[eligible]])).mean()


def atomic_torch(path,obj):
    path = Path(path)
    temp = path.with_suffix('.tmp')
    torch.save(obj,temp)
    temp.replace(path)


def load_model(path):
    saved = torch.load(path,map_location='cpu',weights_only=True)
    model = Ranker(saved['numeric_dim'],saved['summary_dim'],saved['experiment'])
    model.load_state_dict(saved['state'])
    model.eval()
    return model


def predict_arrays(model,graph,layout,summary,batch):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0,len(layout),batch):
            predictions.append(model(graph,torch.from_numpy(layout[start:start+batch]),
                                      torch.from_numpy(summary[start:start+batch])).numpy())
    return np.concatenate(predictions)


def evaluate(model,job,records,split,save=True,early=False):
    rows,frames = [],[]
    for record in records:
        if record['split'] != split:
            continue
        with np.load(Path(job['scratch'])/record['artifact'],allow_pickle=False) as data:
            graph,layout,summary = arrays(data)
            y,ids = data['y'],data['ids']
            if early:
                chosen = np.sort(np.random.default_rng(331).choice(len(y),min(len(y),job['cfg']['gnn_early_configs']),replace=False))
                layout,summary,y,ids = layout[chosen],summary[chosen],y[chosen],ids[chosen]
            pred = predict_arrays(model,graph,layout,summary,job['cfg']['gnn_predict_batch'])
            rows.append(dict(collection=job['collection'],split=split,graph=record['graph'],family=record['family'],
                experiment=job['experiment']['name'],seed=job['seed'],configurations=len(y),
                total_configurations=record['count'],**scores(y,pred)))
            if save:
                frames.append(pd.DataFrame(dict(graph=record['graph'],config_id=ids,runtime=y,prediction=pred)))
    if not rows:
        raise ValueError('No evaluation graphs')
    if save:
        pd.DataFrame(rows).to_csv(Path(job['output'])/f'{split}_graphs.csv',index=False)
        pd.concat(frames).to_csv(Path(job['output'])/f'{split}_predictions.csv.gz',index=False,compression='gzip')
    return float(np.mean([r['score'] for r in rows]))


def train(job):
    cfg,exp,seed = job['cfg'],job['experiment'],job['seed']
    torch.set_num_threads(cfg['cpu_threads'])
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)
    output = Path(job['output']);output.mkdir(parents=True,exist_ok=True)
    records = json.loads((Path(job['scratch'])/'prepared.json').read_text())
    training = training_records(records,dict(matched=True,graph_fraction=1.),seed,cfg)
    with np.load(Path(job['scratch'])/training[0]['artifact']) as data:
        graph,layout,summary = arrays(data)
        numeric_dim,summary_dim = graph[0].shape[1],summary.shape[1]
    del graph,layout,summary
    model = Ranker(numeric_dim,summary_dim,exp)
    optimizer = torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=1e-4)
    start = time.monotonic()
    best,stale,first_epoch,best_epoch = -float('inf'),0,1,0
    history = []
    best_state = None
    previous_seconds = 0.
    checkpoint = output/'latest.pt'
    if checkpoint.exists():
        state = torch.load(checkpoint,map_location='cpu',weights_only=True)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        best,stale,first_epoch,best_epoch = state['best'],state['stale'],state['epoch']+1,state['best_epoch']
        torch.set_rng_state(state['rng'])
        history = state['history']
        previous_seconds = state.get('elapsed_seconds',0.)
        best_state=state['best_state']
        atomic_torch(output/'model.pt',dict(state=best_state,experiment=exp,
                     numeric_dim=numeric_dim,summary_dim=summary_dim))
    train_ids = []
    for record in training:
        ids = training_indices(len(record['indices']),record['graph'],seed,cfg,matched=True)
        train_ids.append(dict(graph=record['graph'],config_ids=np.asarray(record['indices'])[ids].tolist()))
    write_json(output/'training_manifest.json',train_ids)
    for epoch in range(first_epoch,cfg['gnn_epochs']+1):
        if stale >= cfg['gnn_patience']:
            break
        rng = np.random.default_rng(seed+epoch*1009)
        losses=[]
        for index in rng.permutation(len(training)):
            record = training[index]
            with np.load(Path(job['scratch'])/record['artifact'],allow_pickle=False) as data:
                graph,layout,summary = arrays(data)
                ids = training_indices(len(data['y']),record['graph'],seed,cfg,matched=True)
                target = np.log1p(data['y'][ids])
                target = (target-np.median(target)).astype(np.float32)
                order = rng.permutation(len(ids))
                model.train()
                for pos in range(0,len(order),cfg['gnn_batch']):
                    batch = order[pos:pos+cfg['gnn_batch']]
                    chosen=ids[batch]
                    pred=model(graph,torch.from_numpy(layout[chosen]),torch.from_numpy(summary[chosen]))
                    loss=loss_function(pred,torch.from_numpy(target[batch]),exp['loss'])
                    if not torch.isfinite(loss):
                        raise ValueError('Non-finite GNN loss')
                    optimizer.zero_grad(set_to_none=True);loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(),5.)
                    optimizer.step();losses.append(float(loss.detach()))
        dev=evaluate(model,job,records,'dev',save=False,early=True)
        if dev > best + 1e-4:
            best,stale,best_epoch=dev,0,epoch
            best_state=copy.deepcopy(model.state_dict())
            atomic_torch(output/'model.pt',dict(state=best_state,experiment=exp,
                         numeric_dim=numeric_dim,summary_dim=summary_dim))
        else:
            stale+=1
        history.append(dict(epoch=epoch,loss=float(np.mean(losses)),early_dev_score=dev,best_epoch=best_epoch))
        pd.DataFrame(history).to_csv(output/'history.csv',index=False)
        atomic_torch(checkpoint,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),
                     best=best,stale=stale,epoch=epoch,best_epoch=best_epoch,
                     history=history,rng=torch.get_rng_state(),best_state=best_state,
                     elapsed_seconds=previous_seconds+time.monotonic()-start))
        print(f'{exp["name"]} seed {seed} epoch {epoch}: dev={dev:.4f}, best={best:.4f}',flush=True)
    fit_seconds=previous_seconds+time.monotonic()-start
    model=load_model(output/'model.pt')
    dev=evaluate(model,job,records,'dev')
    write_json(output/'result.json',dict(status='ok',experiment=exp['name'],seed=seed,
               collection=job['collection'],dev_score=dev,train_rows=sum(len(r['config_ids']) for r in train_ids),
               feature_count=numeric_dim+36,parameters=sum(p.numel() for p in model.parameters()),
               best_epoch=best_epoch,epochs=len(history),fit_seconds=fit_seconds,
               total_seconds=previous_seconds+time.monotonic()-start,version='3.0.0'))


def holdout(job):
    torch.set_num_threads(job['cfg']['cpu_threads'])
    records=json.loads((Path(job['scratch'])/'prepared.json').read_text())
    score=evaluate(load_model(Path(job['output'])/'model.pt'),job,records,'holdout')
    write_json(Path(job['output'])/'holdout_result.json',dict(status='ok',score=score))


def probe(job):
    """Label-free capacity pilot: one full-size batch, real retained graph, dummy targets."""
    from study_engine import graph_context,layout_context,layout_features,selected_rows,metadata
    cfg=job['cfg'];torch.set_num_threads(cfg['cpu_threads']);torch.manual_seed(42)
    rows=[]
    for filename in job['paths']:
        path=Path(filename);context=graph_context(path,cfg);local=layout_context(path,context,cfg)
        compact=compact_graph(context,local,cfg)
        graph=tuple(torch.from_numpy(compact[k].copy()) for k in
                    ['gnn_numeric','gnn_opcode','gnn_src','gnn_dst','gnn_config_nodes'])
        n=metadata(path,'node_config_feat')[0][0]
        layouts=[];summaries=[]
        for _,row in selected_rows(path,'node_config_feat',list(range(min(n,cfg['gnn_batch'])))):
            features=layout_features(row,context,local,cfg)
            layouts.append(row[compact['gnn_positions']].astype(np.int8))
            summaries.append(np.concatenate([features[k] for k in ['position','opcode','depth','edge']]))
        while len(layouts)<cfg['gnn_batch']:
            layouts.append(layouts[0]);summaries.append(summaries[0])
        layout=torch.from_numpy(np.stack(layouts));summary=torch.from_numpy(np.stack(summaries))
        exp=catalog()[3];model=Ranker(graph[0].shape[1],summary.shape[1],exp)
        optimizer=torch.optim.AdamW(model.parameters(),lr=.001)
        start=time.monotonic()
        for _ in range(3):
            optimizer.zero_grad(set_to_none=True)
            loss=loss_function(model(graph,layout,summary),torch.linspace(-1,1,len(layout)),'pairwise')
            loss.backward();optimizer.step()
        rows.append(dict(graph=path.stem,seconds_per_batch=(time.monotonic()-start)/3,
                         coverage=compact['gnn_coverage'].tolist(),isolated_fraction=float(compact['gnn_isolated_fraction'][0])))
        del model,optimizer,graph,layout,summary,context,local,compact
        release_memory()
    write_json(Path(job['output'])/'probe.json',rows)
'''
(RUNTIME_DIR / "study_gnn.py").write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded implementation: common.py
SOURCE = r'''import importlib.metadata

import json

import os

from pathlib import Path

import shutil

import subprocess

import sys

import time

import zipfile

import numpy as np

import pandas as pd

import psutil

from study_engine import (BLOCKS, COLLECTIONS, GIB, VERSION, digest, experiments,
                          make_manifest, release_memory, scores, stable_seed, write_json)

def default_config():
    return dict(
        collections=COLLECTIONS.copy(), seeds=[42, 137, 2026], split_seed=729,
        layout_train_graphs=40, tile_train_graphs=160, dev_graphs=12, holdout_graphs=24,
        train_pool_configs=384, fit_configs=256, eval_configs=1024,
        opcode_bins=128, depth_bins=64, edge_bins=64, edge_cap=100000,
        hgb_iterations=180, extra_trees=80, mlp_epochs=60, pairs_per_graph=256,
        cpu_threads=2, max_graph_features_mib=96, max_matrix_mib=256,
        worker_ram_gib=5.0, system_reserve_gib=2.0, scratch_limit_gib=2.0,
        output_limit_gib=4.0, disk_reserve_gib=3.0,
        job_minutes=25, extraction_minutes=150, session_hours=9.0,
        bootstrap_repeats=1000, max_excluded_fraction=.1,
        clean_scratch=True, retry_failed=False,
        gnn_collections=['layout:xla:default', 'layout:xla:random'],
        gnn_nodes=512, gnn_config_nodes=384, gnn_edges=4096,
        gnn_train_graphs=24, gnn_configs=64, gnn_batch=16, gnn_predict_batch=32,
        gnn_epochs=20, gnn_patience=5, gnn_early_configs=128, gnn_job_minutes=120,
        inference_batch=128, inference_minutes=90, generate_submissions=True,
        sample_submission='', optional_screening=True,
    )

def scientific_config(cfg):
    resource = {'worker_ram_gib', 'system_reserve_gib', 'scratch_limit_gib', 'output_limit_gib',
                'disk_reserve_gib', 'job_minutes', 'extraction_minutes', 'session_hours',
                'clean_scratch', 'retry_failed', 'gnn_job_minutes', 'inference_minutes', 'sample_submission'}
    return {k: v for k, v in cfg.items() if k not in resource}

def validate_config(cfg):
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct training seeds')
    if cfg['fit_configs'] > cfg['train_pool_configs']:
        raise ValueError('fit_configs cannot exceed train_pool_configs')
    for name in ['fit_configs', 'eval_configs', 'train_pool_configs', 'layout_train_graphs',
                 'tile_train_graphs', 'dev_graphs', 'holdout_graphs', 'hgb_iterations', 'cpu_threads',
                 'gnn_nodes','gnn_config_nodes','gnn_edges','gnn_train_graphs','gnn_configs',
                 'gnn_batch','gnn_predict_batch','gnn_epochs','gnn_patience','gnn_early_configs','inference_batch']:
        if cfg[name] < 1:
            raise ValueError(f'{name} must be positive')
    if cfg['gnn_configs'] > cfg['train_pool_configs']:
        raise ValueError('GNN configurations exceed training pool')
    if set(cfg['gnn_collections']) - {'layout:xla:default','layout:xla:random'}:
        raise ValueError('This GNN protocol targets the two XLA layout collections')
    if set(cfg['collections']) - set(COLLECTIONS):
        raise ValueError('Unknown collection')
    if cfg['worker_ram_gib'] + cfg['system_reserve_gib'] > psutil.virtual_memory().total / GIB:
        raise ValueError('Worker cap plus system reserve exceeds machine RAM')

def folder_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())

def terminate(process):
    try:
        parent = psutil.Process(process.pid)
        children = parent.children(recursive=True)
        for p in children:
            p.terminate()
        parent.terminate()
        _, alive = psutil.wait_procs(children + [parent], timeout=3)
        for p in alive:
            p.kill()
    except psutil.NoSuchProcess:
        pass
    except (psutil.AccessDenied, PermissionError):
        # Still reap our own worker if host process enumeration is restricted.
        process.terminate()
    process.wait(timeout=10)

def execute(job, engine_path, cfg, deadline, status_path, heartbeat=None):
    status_path = Path(status_path)
    if status_path.exists():
        previous = json.loads(status_path.read_text())
        if previous['status'] == 'ok':
            return True
        resumable_timeout = (previous['status']=='timeout' and job.get('experiment',{}).get('model')=='gnn'
                             and (status_path.parent/'latest.pt').exists())
        if not cfg['retry_failed'] and previous['status'] not in ['session_budget', 'interrupted', 'running'] and not resumable_timeout:
            return False
    if time.monotonic() >= deadline:
        return None
    if shutil.disk_usage(status_path.parent).free < cfg['disk_reserve_gib'] * GIB:
        write_json(status_path, dict(status='disk_guard', detail='Disk reserve reached'))
        return False
    job_path = status_path.with_suffix('.job.json')
    log_path = status_path.with_suffix('.log')
    write_json(job_path, job)
    environment = os.environ.copy()
    for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
        environment[name] = str(cfg['cpu_threads'])
    start, peak, reason = time.monotonic(), 0, None
    maximum = cfg['extraction_minutes'] if job['action'] == 'prepare' else cfg['job_minutes']
    if job.get('experiment', {}).get('model') == 'gnn':
        maximum = cfg['gnn_job_minutes']
    if job['action']=='probe':maximum=cfg.get('probe_minutes',5)
    if job['action'] == 'inference':
        maximum = cfg['inference_minutes']
    write_json(status_path,dict(status='running'))
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable, str(engine_path), '--job', str(job_path)],
                                   stdout=log, stderr=subprocess.STDOUT, env=environment)
        try:
            last_message = start
            while process.poll() is None:
                try:
                    parent = psutil.Process(process.pid)
                    rss = sum(p.memory_info().rss for p in [parent] + parent.children(recursive=True)
                              if p.is_running())
                    peak = max(peak, rss)
                except psutil.NoSuchProcess:
                    continue
                elapsed = time.monotonic() - start
                if rss > cfg['worker_ram_gib'] * GIB:
                    reason = 'memory_guard'
                elif psutil.virtual_memory().available < cfg['system_reserve_gib'] * GIB:
                    reason = 'system_memory_guard'
                elif elapsed > maximum * 60:
                    reason = 'timeout'
                elif time.monotonic() >= deadline:
                    reason = 'session_budget'
                elif shutil.disk_usage(status_path.parent).free < cfg['disk_reserve_gib'] * GIB:
                    reason = 'disk_guard'
                if reason:
                    terminate(process)
                    break
                if time.monotonic() - last_message >= 45:
                    print(f'  still running: {elapsed/60:.1f} min, worker {rss/GIB:.2f} GiB', flush=True)
                    last_message = time.monotonic()
                if heartbeat:heartbeat()
                time.sleep(.5)
        except BaseException:
            terminate(process)
            write_json(status_path, dict(status='interrupted', peak_rss_gib=peak/GIB))
            raise
    status = reason or ('ok' if process.returncode == 0 else 'failed')
    write_json(status_path, dict(status=status, peak_rss_gib=peak/GIB,
                                wall_seconds=time.monotonic()-start, exit_code=process.returncode))
    release_memory()
    if status != 'ok':
        print(f'  {status}: {log_path}', flush=True)
    return None if status == 'session_budget' else status == 'ok'

def aligned_predictions(output, names, seeds, split):
    """One graph/config ID alignment for all ranks; reject mismatches instead of intersecting."""
    result = {}
    for name in names:
        frames = []
        for seed in seeds:
            path = Path(output) / 'trials' / name / str(seed) / f'{split}_predictions.csv.gz'
            frame = pd.read_csv(path).sort_values(['graph', 'config_id']).reset_index(drop=True)
            if frames and not frame[['graph', 'config_id', 'runtime']].equals(frames[0][['graph', 'config_id', 'runtime']]):
                raise ValueError('Prediction IDs or labels differ across seeds')
            frame['rank'] = frame.groupby('graph').prediction.rank(method='average', pct=True)
            frames.append(frame)
        mean = frames[0][['graph', 'config_id', 'runtime']].copy()
        mean['prediction'] = np.mean([f['rank'].to_numpy() for f in frames], axis=0)
        if result:
            first = next(iter(result.values()))
            if not mean[['graph', 'config_id', 'runtime']].equals(first[['graph', 'config_id', 'runtime']]):
                raise ValueError('Prediction IDs differ between methods')
        result[name] = mean
    return result

def frame_score(frame, tile):
    return float(np.mean([scores(g.runtime.to_numpy(), g.prediction.to_numpy(), tile)['score']
                          for _, g in frame.groupby('graph')]))

def paired_summary(graphs, seeds, repeats, reference='B0_basic_hgb'):
    if graphs.empty:
        return pd.DataFrame()
    baseline = graphs[graphs.experiment == reference][['graph', 'seed', 'score']].rename(columns={'score':'baseline'})
    results = []
    for name, frame in graphs.groupby('experiment'):
        paired = frame.merge(baseline, on=['graph','seed'], validate='one_to_one')
        paired['delta'] = paired.score - paired.baseline
        # Average repeats within graph; bootstrap independent graphs, not configs/seeds.
        by_graph = paired.groupby('graph').agg(delta=('delta','mean'), seeds=('seed','nunique'))
        by_graph = by_graph[by_graph.seeds == len(seeds)]
        if by_graph.empty:
            continue
        d = by_graph.delta.to_numpy()
        rng = np.random.default_rng(551)
        boot = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(repeats)])
        low, high = np.percentile(boot, [2.5,97.5])
        base_mean = float(paired.baseline.mean())
        candidate_mean = float(paired.score.mean())
        results.append(dict(experiment=name, reference=reference, baseline_score=base_mean,
                            candidate_score=candidate_mean, mean_paired_gain=float(d.mean()),
                            score_points=100*float(d.mean()),
                            relative_gain_pct=(100*float(d.mean())/base_mean if base_mean > .01 else None),
                            graphs=len(d),
                            graph_win_fraction=float(np.mean(d > 0)),
                            bootstrap_low=float(low), bootstrap_high=float(high),
                            uncertainty_note='exploratory graph bootstrap; no multiplicity correction'))
    return pd.DataFrame(results)
'''
(RUNTIME_DIR / "study_common.py").write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded implementation: plan.py
SOURCE = r'''"""Predeclared v3 experiments and graph-disjoint development folds."""
import json
from pathlib import Path
import zipfile
import numpy as np
from study_engine import (COLLECTIONS,balanced_order,metadata,stable_seed,digest,family,
                          experiments)
from study_common import default_config as previous_defaults


def default_config():
    cfg=previous_defaults()
    cfg.update(version='3.0.0',folds=[0,1],dev_graphs=12,split_seed=729,
        graph_budgets=[512,1024,2048,4096],data_graph_counts=[24,40,0],data_config_counts=[64,128,256],
        gnn_epochs=20,gnn_patience=5,gnn_batch=16,gnn_predict_batch=16,
        gnn_sampler='uniform',max_matrix_mib=512,scratch_limit_gib=4.,output_limit_gib=12.,
        disk_reserve_gib=4.,probe_minutes=8,probe_ram_gib=3.5,probe_graphs=3,
        layout_more_graphs=128,tile_more_graphs=320,large_fit_configs=384,
        confirmation_top_k=2,checkpoint_minutes=30,session_hours=9.,
        fold_regression_tolerance=.02,sample_submission='',generate_submissions=True)
    return cfg


def profile(nodes,sampler='uniform'):
    return dict(nodes=int(nodes),config_nodes=int(nodes*3//4),edges=int(nodes*8),sampler=sampler)


def effective_config(cfg,exp,collection):
    result=dict(cfg)
    p=exp.get('profile')
    if p:
        result.update(gnn_nodes=p['nodes'],gnn_config_nodes=p['config_nodes'],
                      gnn_edges=p['edges'],gnn_sampler=p['sampler'])
    if exp['model']=='gnn' or exp.get('matched'):
        result.update(gnn_train_graphs=exp.get('train_graphs',24) or 1000000,
                      gnn_configs=exp.get('configs',64))
    else:
        result['fit_configs']=exp.get('configs',256)
    if exp['model']!='gnn':result['gnn_collections']=[]
    return result


def manifest(root,collection,cfg,fold):
    root=Path(root);folder=root.joinpath(*collection.split(':'))
    order=balanced_order(list((folder/'train').glob('*.npz')),cfg['split_seed'])
    ndev=min(cfg['dev_graphs'],max(2,len(order)//5))
    if len(order)<ndev*len(cfg['folds'])+3:
        raise ValueError('Insufficient training graphs for disjoint development folds')
    # Each fold has a disjoint development block. Training may include another fold's dev graphs;
    # those models are NEVER used to predict that other fold during selection.
    slot=cfg['folds'].index(fold)
    dev=order[slot*ndev:(slot+1)*ndev]
    available=[p for p in order if p not in dev]
    if collection in cfg['gnn_collections']:
        train=available
    else:
        cap=cfg['tile_more_graphs'] if collection.startswith('tile') else cfg['layout_more_graphs']
        train=available[:max(cap,cfg['tile_train_graphs'] if collection.startswith('tile') else cfg['layout_train_graphs'])]
    holdout=balanced_order(list((folder/'valid').glob('*.npz')),cfg['split_seed']+1)[:cfg['holdout_graphs']]
    if {p.stem for p in order}&{p.stem for p in holdout}:raise ValueError('Train/valid graph name overlap')
    records=[]
    for split,paths in [('train',train),('dev',dev),('holdout',holdout)]:
        for path in sorted(paths):
            key='config_feat' if collection.startswith('tile') else 'node_config_feat'
            shape,fortran,dtype=metadata(path,key);n=shape[0]
            cap=cfg['train_pool_configs'] if split=='train' else cfg['eval_configs']
            # Fold-independent ordering gives the same candidates whenever a graph has the same role.
            ids=np.sort(np.random.default_rng(stable_seed(f'{cfg["split_seed"]}/{collection}/{path.stem}'))
                        .choice(n,min(n,cap),replace=False))
            with zipfile.ZipFile(path) as z:source=[(i.filename,i.CRC,i.file_size) for i in z.infolist()]
            records.append(dict(split=split,graph=path.stem,family=family(path.stem),
                relative_path=str(path.relative_to(root)),count=n,indices=ids.tolist(),
                source=digest(source),fortran=bool(fortran),row_bytes=int(np.prod(shape[1:]))*dtype.itemsize))
    return dict(collection=collection,fold=fold,records=records,
                split_note='graph-disjoint development within fold; official valid is an already-inspected audit')


def gnn_exp(name,recipe,nodes=512,sampler='uniform',graphs=24,configs=64,stage='coverage',**changes):
    from study_gnn import catalog
    exp=dict(next(e for e in catalog() if e['name']==recipe))
    exp.update(name=name,profile=profile(nodes,sampler),train_graphs=graphs,configs=configs,stage=stage,**changes)
    return exp


def coverage_plan(cfg):
    result=[]
    for nodes in cfg['graph_budgets']:
        result.extend([gnn_exp(f'G_reference_n{nodes}','G0_reference',nodes),
                       gnn_exp(f'G_full_n{nodes}','G_full',nodes)])
    return result


def followup_plan(cfg,selected_profile):
    n=selected_profile['nodes'];result=[]
    for suffix,recipe in [('masks','G_add_masks'),('context','G_add_context'),('no_messages','G_no_messages')]:
        result.append(gnn_exp(f'G_{suffix}_n{n}',recipe,n,stage='component_controls'))
    result.append(gnn_exp(f'G_connected_n{n}','G_full',n,sampler='connected',stage='sampler_control'))
    full=next(e for e in experiments(False) if e['name']=='P_full')
    for graphs in cfg['data_graph_counts']:
        for count in cfg['data_config_counts']:
            tag=f'g{graphs or "all"}_c{count}'
            if graphs!=24 or count!=64:
                result.append(gnn_exp(f'G_data_{tag}','G_full',n,graphs=graphs,configs=count,stage='training_data'))
            result.append(dict(full,name=f'H_matched_{tag}',train_graphs=graphs,configs=count,
                               matched=True,stage='matched_training_data'))
    return result


def tabular_plan(collection,cfg):
    tile=collection.startswith('tile');catalog=experiments(tile)
    base=next(e for e in catalog if e['name']=='B0_basic_hgb')
    full=next(e for e in catalog if e['name']=='P_full')
    small=cfg['tile_train_graphs'] if tile else cfg['layout_train_graphs']
    large=cfg['tile_more_graphs'] if tile else cfg['layout_more_graphs']
    def trial(name,exp,graphs,configs,**changes):
        return dict(exp,name=name,train_graphs=graphs,configs=configs,stage='tabular',**changes)
    return [trial('B0_basic_hgb',base,small,cfg['fit_configs']),
            trial('T_full_small',full,small,cfg['fit_configs']),
            trial('T_more_graphs',full,large,cfg['fit_configs']),
            trial('T_more_configs',full,small,cfg['large_fit_configs']),
            trial('T_more_both',full,large,cfg['large_fit_configs']),
            trial('T_deeper_hgb',full,large,cfg['fit_configs'],iterations=300,leaves=63,min_leaf=10,learning_rate=.04),
            trial('T_extra_trees',full,large,cfg['fit_configs'],model='extra_trees')]
'''
(RUNTIME_DIR / "study_plan.py").write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded implementation: submission.py
SOURCE = r'''"""Full-candidate streamed prediction with frozen development-selected models."""
import json
from pathlib import Path
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from threadpoolctl import threadpool_limits

from study_engine import (metadata, graph_context, layout_context, layout_features, tile_features,
                          selected_rows, predict, guard, digest, write_json, COLLECTIONS)


def ranking(prediction, count, tile, template_values):
    n = len(prediction)
    if not n or not np.isfinite(prediction).all():
        raise ValueError('Empty or nonfinite predictions')
    ordered = np.argsort(prediction,kind='stable')[:min(n,count)].tolist()
    if not tile and count > n:
        raise ValueError('Layout template requests more candidates than exist')
    if count > n:
        # Preserve ONLY the supplied template suffix, never invent candidate IDs.
        # The competition template has 0;1;2;3;4 for some tile graphs with <5 candidates.
        expected = list(range(n,count))
        if not tile or template_values != list(range(count)):
            raise ValueError('Unrecognized short-tile template contract')
        ordered += expected
    validate_ranking(ordered,n,count,tile,template_values)
    return ordered


def validate_ranking(values,n,count,tile,template_values):
    if len(values)!=count or len(set(values))!=count:
        raise ValueError('Ranking length/uniqueness mismatch')
    prefix=values[:min(n,count)]
    if any(v<0 or v>=n for v in prefix):
        raise ValueError('Invalid candidate in ranking')
    if not tile and (count>n or (count==n and sorted(values)!=list(range(n)))):
        raise ValueError('Layout ranking is not a full permutation')
    if count>n and (not tile or template_values!=list(range(count)) or values[n:]!=template_values[n:]):
        raise ValueError('Short tile suffix differs from original template')


def infer_graph(job):
    from study_plan import effective_config
    cfg=job['cfg'];tile=job['collection'].startswith('tile')
    path=Path(job['path']);directory=Path(job['directory']);lock=job['lock']
    names=list(dict.fromkeys(['B0_basic_hgb',lock['winner']]+list(lock['components'])))
    catalog={e['name']:e for e in json.loads((directory/'experiment_plan.json').read_text())}
    members=[];profiles={}
    for name in names:
        for fold in cfg['folds']:
            for seed in cfg['seeds']:
                model_dir=directory/'folds'/str(fold)/'trials'/name/str(seed)
                if catalog[name]['model']=='gnn':
                    import torch
                    from study_gnn import load_model
                    torch.set_num_threads(cfg['cpu_threads'])
                    model=load_model(model_dir/'model.pt');key=digest(catalog[name]['profile'])
                    profiles[key]=effective_config(cfg,catalog[name],job['collection'])
                else:model=joblib.load(model_dir/'model.joblib');key=None
                members.append((name,model,key))
    n=metadata(path,'config_feat' if tile else 'node_config_feat')[0][0]
    guard(n*8*(len(members)+8));predictions=np.empty((len(members),n),np.float64)
    context=graph_context(path,cfg);local=None if tile else layout_context(path,context,cfg)
    bounded={};graphs={}
    if profiles:
        from study_gnn import compact_graph,predict_arrays
        for key,settings in profiles.items():
            bounded[key]=compact_graph(context,local,settings)
            graphs[key]=tuple(torch.from_numpy(bounded[key][k].copy()) for k in
                          ['gnn_numeric','gnn_opcode','gnn_src','gnn_dst','gnn_config_nodes'])
    features=[];layouts={key:[] for key in profiles};indices=[]
    def flush():
        data={k:np.stack([row[k] for row in features]) for k in features[0]};xs={}
        if profiles:
            summary=np.concatenate([data[k] for k in ['position','opcode','depth','edge']],axis=1)
            batch_layout={k:np.stack(v) for k,v in layouts.items()}
        for member,(name,model,key) in enumerate(members):
            if key is not None:pred=predict_arrays(model,graphs[key],batch_layout[key],summary,cfg['gnn_predict_batch'])
            else:
                blocks=tuple(model['experiment']['blocks'])
                if blocks not in xs:xs[blocks]=np.concatenate([data[k] for k in blocks],axis=1)
                pred=predict(model,xs[blocks])
            predictions[member,indices]=pred
        features.clear();indices.clear()
        for values in layouts.values():values.clear()
    with threadpool_limits(limits=cfg['cpu_threads']):
        for index,row in selected_rows(path,'config_feat' if tile else 'node_config_feat',np.arange(n)):
            features.append(tile_features(row,context,cfg) if tile else layout_features(row,context,local,cfg))
            for key,graph in bounded.items():layouts[key].append(row[graph['gnn_positions']].astype(np.int8))
            indices.append(index)
            if len(indices)>=cfg['inference_batch']:flush()
        if indices:flush()
    by_method={name:np.mean([rankdata(predictions[i],method='average')/n for i,(member,_,_) in enumerate(members)
                           if member==name],axis=0) for name in names}
    chosen=sum(w*by_method[name] for name,w in lock['components'].items());count=len(job['template_values'])
    result=dict(ID=job['id'],count=n,components=lock['components'],folds=cfg['folds'],seeds=cfg['seeds'],
                gnn_coverage={k:v['gnn_coverage'].tolist() for k,v in bounded.items()},
                baseline=ranking(by_method['B0_basic_hgb'],count,tile,job['template_values']),
                tabular=ranking(by_method[lock['winner']],count,tile,job['template_values']),
                selected=ranking(chosen,count,tile,job['template_values']))
    write_json(Path(job['output'])/'prediction.json',result)


def locate_template(root,explicit):
    if explicit:
        path=Path(explicit)
        if not path.is_file():raise FileNotFoundError(path)
        return path
    for folder in [root,root.parent,root.parent.parent,root.parent.parent.parent]:
        for name in ['sample_submission.csv','sample_submission_Eugene.csv']:
            if (folder/name).is_file():return folder/name
    raise FileNotFoundError('Set SAMPLE_SUBMISSION to the competition template CSV path')


def preflight(root,cfg):
    """Check the complete test/template contract before spending hours on training."""
    root=Path(root)
    template=pd.read_csv(locate_template(root,cfg['sample_submission']),dtype=str)
    if list(template.columns)!=['ID','TopConfigs'] or template.ID.duplicated().any() or template.isna().any().any():
        raise ValueError('Invalid sample submission template')
    if {value.rsplit(':',1)[0] for value in template.ID}!=set(cfg['collections']):
        raise ValueError('Template must cover exactly the configured collections')
    short,truncated=0,0
    for row in template.itertuples(index=False):
        collection,stem=row.ID.rsplit(':',1)
        if collection not in COLLECTIONS or '/' in stem or '\\' in stem:
            raise ValueError('Invalid template ID')
        tile=collection.startswith('tile')
        path=root.joinpath(*collection.split(':'),'test',stem+'.npz')
        shape,fortran,_=metadata(path,'config_feat' if tile else 'node_config_feat')
        if not shape[0] or fortran:
            raise ValueError(f'{row.ID}: empty or non-streamable test configurations')
        values=[int(v) for v in row.TopConfigs.split(';')]
        validate_ranking(values,shape[0],len(values),tile,values)
        short+=tile and len(values)>shape[0]
        truncated+=not tile and len(values)<shape[0]
    return dict(rows=len(template),short_tile_rows=short,layout_rows_truncated_to_template=truncated)


def export_submissions(root,output,engine_path,cfg,deadline,execute):
    template_path=locate_template(root,cfg['sample_submission'])
    template=pd.read_csv(template_path,dtype=str)
    if list(template.columns)!=['ID','TopConfigs'] or template.ID.duplicated().any() or template.isna().any().any():
        raise ValueError('Invalid sample submission template')
    actual_collections={row.rsplit(':',1)[0] for row in template.ID}
    if actual_collections!=set(cfg['collections']):
        raise ValueError('Template collections and configured collections differ')
    tests=[]
    for row in template.itertuples(index=False):
        collection,stem=row.ID.rsplit(':',1)
        if '/' in stem or '\\' in stem or collection not in COLLECTIONS:
            raise ValueError('Invalid test ID')
        path=root.joinpath(*collection.split(':'),'test',stem+'.npz')
        with zipfile.ZipFile(path) as z:
            fingerprint=[(i.filename,i.CRC,i.file_size) for i in z.infolist()]
        tests.append(dict(id=row.ID,collection=collection,relative_path=str(path.relative_to(root)),
                          template_values=[int(v) for v in row.TopConfigs.split(';')],fingerprint=fingerprint))
    # JSON round trip normalizes tuples for resume comparison.
    manifest=json.loads(json.dumps(dict(template_sha=digest(template_path.read_text()),tests=tests)))
    marker=output/'test_manifest.json'
    if marker.exists() and json.loads(marker.read_text())!=manifest:
        raise ValueError('Test inputs/template changed; use a new RUN_NAME')
    write_json(marker,manifest)
    successes=[]
    for i,record in enumerate(tests):
        directory=output/record['collection'].replace(':','_')
        lock=json.loads((directory/'selection_lock.json').read_text())
        destination=output/'test_predictions'/f'{i:05d}';destination.mkdir(parents=True,exist_ok=True)
        job=dict(action='inference',cfg=cfg,lock=lock,directory=str(directory),
                 path=str(root/record['relative_path']),output=str(destination),**record)
        print(f'Test graph {i+1}/{len(tests)}: {record["id"]}',flush=True)
        ok=execute(job,engine_path,cfg,deadline,destination/'status.json')
        if ok:successes.append(json.loads((destination/'prediction.json').read_text()))
        if ok is None:break
    if len(successes)!=len(tests):
        write_json(output/'submission_status.json',dict(status='incomplete',completed=len(successes),total=len(tests)))
        return
    for kind in ['baseline','tabular','selected']:
        rows=[]
        for record,result in zip(tests,successes):
            if result['ID']!=record['id']:raise ValueError('Prediction ID mismatch')
            validate_ranking(result[kind],result['count'],len(record['template_values']),
                             record['collection'].startswith('tile'),record['template_values'])
            rows.append(dict(ID=result['ID'],TopConfigs=';'.join(map(str,result[kind]))))
        frame=pd.DataFrame(rows)
        if not frame.ID.equals(template.ID):raise ValueError('Submission order mismatch')
        target=output/f'submission_{kind}.csv';temp=target.with_suffix('.csv.tmp')
        frame.to_csv(temp,index=False);temp.replace(target)
    write_json(output/'submission_status.json',dict(status='complete',rows=len(tests),
               template=str(template_path.name),full_candidate_inference=True,
               layout_truncated_to_template=sum(not r['collection'].startswith('tile') and len(r['template_values'])<s['count'] for r,s in zip(tests,successes)),
               short_tile_template_rows=sum(len(r['template_values'])>s['count'] for r,s in zip(tests,successes)),
               fitted_models='same frozen fold/seed models used in audit; no holdout retraining'))
'''
(RUNTIME_DIR / "study_submission.py").write_text(SOURCE)
del SOURCE

In [ ]:
# Embedded implementation: controller.py
SOURCE = r'''"""V3: two-fold screening, controlled coverage/data comparisons, confirmation and export."""
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import time
import zipfile
import numpy as np
import pandas as pd

from study_engine import GIB,VERSION,digest,write_json,metadata,release_memory
from study_common import execute,folder_bytes,frame_score,aligned_predictions,paired_summary,validate_config
from study_plan import (default_config,manifest,coverage_plan,followup_plan,tabular_plan,
                        effective_config,profile)


def make_bundle(output,reserve_gib=1.):
    output=Path(output);target=output.parent/(output.name+'_results.zip')
    required=folder_bytes(output)*1.05+reserve_gib*GIB
    if shutil.disk_usage(output.parent).free<required:
        raise OSError('Insufficient free disk to build a safe checkpoint ZIP; saved result files remain intact')
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2) as archive:
        for path in sorted(output.rglob('*')):
            if path.is_file() and not path.name.endswith('.tmp'):
                archive.write(path,path.relative_to(output))
    temporary.replace(target)
    return target


def restore_archive(archive_path,output,limit_gib):
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    if any(output.iterdir()):raise ValueError('Restore into an empty result directory only')
    with zipfile.ZipFile(archive_path) as archive:
        if sum(i.file_size for i in archive.infolist())>limit_gib*GIB:
            raise ValueError('Archive exceeds output limit')
        for item in archive.infolist():
            dest=(output/item.filename).resolve()
            if output.resolve() not in dest.parents:raise ValueError('Unsafe archive path')
        if archive.testzip() is not None:raise ValueError('Damaged checkpoint ZIP')
        archive.extractall(output)


def scientific_config(cfg):
    resources={'worker_ram_gib','system_reserve_gib','scratch_limit_gib','output_limit_gib',
               'disk_reserve_gib','job_minutes','extraction_minutes','session_hours','gnn_job_minutes',
               'inference_minutes','checkpoint_minutes','sample_submission','retry_failed','clean_scratch'}
    # Probe thresholds affect scientific eligibility, so they remain part of run identity.
    return {k:v for k,v in cfg.items() if k not in resources}


def trial_dir(directory,fold,exp,seed):
    return Path(directory)/'folds'/str(fold)/'trials'/exp/str(seed)


def finished(directory,exp,cfg,seeds=None,action='trial'):
    seeds=cfg['seeds'] if seeds is None else seeds
    marker='status.json' if action=='trial' else 'holdout_status.json'
    result='result.json' if action=='trial' else 'holdout_result.json'
    return all((trial_dir(directory,f,exp,s)/result).exists() and
               (trial_dir(directory,f,exp,s)/marker).exists() and
               json.loads((trial_dir(directory,f,exp,s)/marker).read_text())['status']=='ok'
               for f in cfg['folds'] for s in seeds)


def prediction_sets(directory,names,cfg,seeds,split):
    return {fold:aligned_predictions(Path(directory)/'folds'/str(fold),names,seeds,split)
            for fold in cfg['folds']}


def method_scores(directory,names,cfg,seeds,collection):
    frames=prediction_sets(directory,names,cfg,seeds,'dev');tile=collection.startswith('tile')
    return {name:[frame_score(frames[f][name],tile) for f in cfg['folds']] for name in names}


def select_coverage(directory,catalog,cfg,collection):
    path=Path(directory)/'coverage_lock.json'
    if path.exists():return json.loads(path.read_text())
    candidates=[e for e in catalog if e['name'].startswith('G_full_') and
                finished(directory,e['name'],cfg,[cfg['seeds'][0]])]
    if not candidates:raise RuntimeError('No complete coverage candidate; inspect failures before resuming')
    values=method_scores(directory,[e['name'] for e in candidates],cfg,[cfg['seeds'][0]],collection)
    # Tie break toward smaller graph budgets; labels only come from development folds.
    best=max(candidates,key=lambda e:(np.mean(values[e['name']]),-e['profile']['nodes']))
    lock=dict(experiment=best['name'],profile=best['profile'],fold_scores=values,
              source='development folds, screening seed only',selected_mean=float(np.mean(values[best['name']])) )
    write_json(path,lock);return lock


def confirmation_plan(directory,catalog,cfg,collection,coverage):
    path=Path(directory)/'confirmation_plan.json'
    if path.exists():return json.loads(path.read_text())
    complete=[e['name'] for e in catalog if finished(directory,e['name'],cfg,[cfg['seeds'][0]])]
    required=['B0_basic_hgb','T_full_small']
    if coverage:
        base=cfg['graph_budgets'][0];chosen=coverage['profile']['nodes']
        required += [f'G_reference_n{base}',f'G_full_n{base}',f'G_reference_n{chosen}',f'G_full_n{chosen}']
        # Confirm the fixed 2×2 corners: data changes can be separated and their interaction measured.
        max_c=max(cfg['data_config_counts']);max_g=0 if 0 in cfg['data_graph_counts'] else max(cfg['data_graph_counts'])
        for g,c in [(24,64),(24,max_c),(max_g,64),(max_g,max_c)]:
            tag=f'g{g or "all"}_c{c}'
            required += [f'G_full_n{chosen}' if (g,c)==(24,64) else f'G_data_{tag}',f'H_matched_{tag}']
        required += [f'G_masks_n{chosen}',f'G_context_n{chosen}',f'G_no_messages_n{chosen}',f'G_connected_n{chosen}']
    scores=method_scores(directory,complete,cfg,[cfg['seeds'][0]],collection)
    for neural in [False,True]:
        group=[n for n in complete if n.startswith('G')==neural]
        required += sorted(group,key=lambda n:np.mean(scores[n]),reverse=True)[:cfg['confirmation_top_k']]
    all_names={e['name'] for e in catalog}
    required=list(dict.fromkeys(n for n in required if n in all_names))
    plan=dict(names=required,screened_complete=complete,screening_fold_scores=scores,
              note='All specified controls plus development-selected leaders; confirmation is not independent selection data')
    write_json(path,plan);return plan


def lock_selection(directory,catalog,cfg,collection,names):
    path=Path(directory)/'selection_lock.json'
    if path.exists():return json.loads(path.read_text())
    complete=[n for n in names if finished(directory,n,cfg)]
    if 'B0_basic_hgb' not in complete:raise RuntimeError('Baseline must finish every fold/seed')
    if set(complete)!=set(names):raise RuntimeError('Confirmation is incomplete; no final selection was made')
    pred=prediction_sets(directory,complete,cfg,cfg['seeds'],'dev');tile=collection.startswith('tile')
    values={n:[frame_score(pred[f][n],tile) for f in cfg['folds']] for n in complete}
    tabular=max([n for n in complete if not n.startswith('G')],key=lambda n:np.mean(values[n]))
    neural=[n for n in complete if n.startswith('G')]
    gnn=max(neural,key=lambda n:np.mean(values[n])) if neural else None
    candidates=[]
    for base in list(dict.fromkeys(['B0_basic_hgb',tabular])):
        for weight in ([0.,.25,.5,.75,1.] if gnn else [0.]):
            components={base:1-weight}
            if gnn:components[gnn]=weight
            components={k:v for k,v in components.items() if v>0}
            fold_scores=[]
            for fold in cfg['folds']:
                frame=pred[fold][base].copy()
                frame['prediction']=sum(w*pred[fold][n].prediction for n,w in components.items())
                fold_scores.append(frame_score(frame,tile))
            delta=np.array(fold_scores)-np.array(values['B0_basic_hgb'])
            candidates.append(dict(components=components,fold_scores=fold_scores,mean_score=float(np.mean(fold_scores)),
                                   eligible=bool(np.min(delta)>=-cfg['fold_regression_tolerance'])))
    selected=max([c for c in candidates if c['eligible']],key=lambda c:c['mean_score'])
    lock=dict(components=selected['components'],winner=tabular,gnn=gnn,confirmed=complete,
        seeds=cfg['seeds'],folds=cfg['folds'],dev_score=selected['mean_score'],fold_scores=selected['fold_scores'],
        method_fold_scores=values,blend_grid=candidates,
        source='development only; each fold uses only models trained without that fold',
        fold_regression_tolerance=cfg['fold_regression_tolerance'],
        locked_at=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()))
    write_json(path,lock);return lock


def factor_contrasts(graphs,cfg,coverage,seeds):
    """Explicit paired changes; each row states the one experimental factor changed."""
    pairs=[]
    def add(factor,reference,candidate):
        if reference!=candidate:pairs.append((factor,reference,candidate))
    for candidate,factor,reference in [
        ('T_full_small','tabular features','B0_basic_hgb'),
        ('T_more_graphs','training graphs','T_full_small'),
        ('T_more_configs','training configurations','T_full_small'),
        ('T_more_both','training configurations','T_more_graphs'),
        ('T_more_both','training graphs','T_more_configs'),
        ('T_deeper_hgb','HGB capacity','T_more_graphs'),
        ('T_extra_trees','tabular estimator','T_more_graphs')]:add(factor,reference,candidate)
    if coverage:
        chosen=coverage['profile']['nodes'];base=cfg['graph_budgets'][0]
        reference=f'G_reference_n{chosen}';full=f'G_full_n{chosen}'
        for n in cfg['graph_budgets']:
            for recipe in ['reference','full']:
                add('graph coverage',f'G_{recipe}_n{base}',f'G_{recipe}_n{n}')
        for recipe,factor in [('masks','mask features'),('context','context features'),('full','masks and context')]:
            add(factor,reference,f'G_{recipe}_n{chosen}')
        add('message passing',f'G_no_messages_n{chosen}',full)
        add('connected sampling',full,f'G_connected_n{chosen}')
        def name(model,g,c):
            return full if model=='G_data' and (g,c)==(24,64) else f'{model}_g{g or "all"}_c{c}'
        # Include fixed endpoint contrasts even when intermediate sizes are screening-only.
        for model in ['G_data','H_matched']:
            for g in cfg['data_graph_counts']:
                for c in cfg['data_config_counts']:
                    add('training configurations',name(model,g,64),name(model,g,c))
                    add('training graphs',name(model,24,c),name(model,g,c))
        for g in cfg['data_graph_counts']:
            for c in cfg['data_config_counts']:
                add('GNN versus matched HGB',name('H_matched',g,c),name('G_data',g,c))
    available=set(graphs.experiment);rows=[]
    for factor,reference,candidate in dict.fromkeys(pairs):
        if {reference,candidate}<=available:
            table=paired_summary(graphs[graphs.experiment.isin([reference,candidate])],
                                 seeds,cfg['bootstrap_repeats'],reference)
            if table.empty:continue
            for row in table[table.experiment==candidate].to_dict('records'):
                rows.append(dict(factor=factor,**row))
    return pd.DataFrame(rows)


def summarize(directory,cfg,collection):
    directory=Path(directory);all_results=[];catalog_path=directory/'experiment_plan.json'
    catalog=json.loads(catalog_path.read_text()) if catalog_path.exists() else []
    for p in directory.glob('folds/*/trials/*/*/result.json'):
        status=p.parent/'status.json'
        if status.exists() and json.loads(status.read_text())['status']=='ok':
            row=json.loads(p.read_text());row.update(json.loads(status.read_text()));row['fold']=int(p.parents[3].name)
            all_results.append(row)
    if all_results:pd.DataFrame(all_results).to_csv(directory/'all_trials.csv',index=False)
    for split in ['dev','holdout']:
        frames=[]
        for p in directory.glob(f'folds/*/trials/*/*/{split}_graphs.csv'):
            marker=p.parent/('status.json' if split=='dev' else 'holdout_status.json')
            if marker.exists() and json.loads(marker.read_text())['status']=='ok':
                f=pd.read_csv(p);f['fold']=int(p.parents[3].name);frames.append(f)
        if not frames:continue
        graphs=pd.concat(frames,ignore_index=True);graphs.to_csv(directory/f'{split}_graph_scores.csv',index=False)
        graphs.groupby(['experiment','fold','seed']).score.mean().reset_index().to_csv(directory/f'{split}_fold_seed_scores.csv',index=False)
        # Screening and confirmed comparisons are labelled separately; never mix seed counts.
        for scope,seeds in [('screening',[cfg['seeds'][0]]),('confirmed',cfg['seeds'])]:
            names=[e['name'] for e in catalog if finished(directory,e['name'],cfg,seeds,'trial' if split=='dev' else 'holdout')]
            subset=graphs[graphs.experiment.isin(names)&graphs.seed.isin(seeds)]
            if subset.empty:continue
            reduced=subset.groupby(['experiment','graph','seed'],as_index=False).score.mean()
            refs=['B0_basic_hgb',f'G_reference_n{cfg["graph_budgets"][0]}',f'G_full_n{cfg["graph_budgets"][0]}']
            lock_path=directory/'coverage_lock.json'
            if lock_path.exists():refs += [json.loads(lock_path.read_text())['experiment'],'H_matched_g24_c64']
            for ref in dict.fromkeys(refs):
                if ref in names:
                    paired_summary(reduced,seeds,cfg['bootstrap_repeats'],ref).to_csv(
                        directory/f'{split}_{scope}_vs_{ref}.csv',index=False)
            coverage=json.loads(lock_path.read_text()) if lock_path.exists() else None
            contrasts=factor_contrasts(reduced,cfg,coverage,seeds)
            if not contrasts.empty:
                contrasts.to_csv(directory/f'{split}_{scope}_factor_contrasts.csv',index=False)
    lock_path=directory/'selection_lock.json'
    if not lock_path.exists():return
    lock=json.loads(lock_path.read_text());names=list(dict.fromkeys(['B0_basic_hgb',lock['winner']]+list(lock['components'])))
    if not all(finished(directory,n,cfg,action='holdout') for n in names):return
    by_fold=prediction_sets(directory,names,cfg,cfg['seeds'],'holdout');pred={}
    for name in names:
        first=by_fold[cfg['folds'][0]][name].copy()
        for fold in cfg['folds'][1:]:
            if not first[['graph','config_id','runtime']].equals(by_fold[fold][name][['graph','config_id','runtime']]):
                raise ValueError('Holdout labels/IDs differ across folds')
        first['prediction']=np.mean([by_fold[f][name].prediction.to_numpy() for f in cfg['folds']],axis=0);pred[name]=first
    blend=pred['B0_basic_hgb'].copy();blend['prediction']=sum(w*pred[n].prediction for n,w in lock['components'].items())
    rows=[]
    for name,frame in [('baseline',pred['B0_basic_hgb']),('tabular',pred[lock['winner']]),('selected',blend)]:
        frame.to_csv(directory/f'{name}_holdout_predictions.csv.gz',index=False,compression='gzip')
        for graph,g in frame.groupby('graph'):
            from study_engine import scores
            rows.append(dict(experiment=name,graph=graph,seed=0,**scores(g.runtime.to_numpy(),g.prediction.to_numpy(),collection.startswith('tile'))))
    table=pd.DataFrame(rows);table.to_csv(directory/'system_holdout_graphs.csv',index=False)
    paired_summary(table,[0],cfg['bootstrap_repeats'],'baseline').to_csv(directory/'system_holdout_gains.csv',index=False)


def write_summary(output,cfg,states):
    output=Path(output)
    write_json(output/'run_state.json',dict(collections=states,config=cfg))
    lines=['# V3 checkpoint','', 'Status refers to completed workers, not a promised runtime or leaderboard gain.',
           'Development folds choose profiles, data recipes and blends. Official valid is an already-inspected audit.',
           'Screening uses the first seed on every fold; key controls and finalists use all seeds.',
           'Fold-specific models only evaluate their own development fold. Test/audit ensembles average all fold/seed models.',
           '','| Collection | Status |','|---|---|']
    lines += [f'| {c} | {state} |' for c,state in states.items()]
    lines += ['', '## Interpretation', '',
        '- `experiment_plan.json`: exact stages, graph caps, training sizes and feature recipes.',
        '- `probes/`: label-free admission pilots; failed budgets are explicit resource exclusions.',
        '- `coverage_lock.json`: development-selected graph budget, before the training-data sweep.',
        '- `confirmation_plan.json`: repeated comparisons, separate from one-seed screening.',
        '- `selection_lock.json`: selected model/blend, including per-fold regression checks.',
        '- `preparations/`: per-profile/fold manifests and actual retained-node/edge/isolation diagnostics.',
        '- `*_confirmed_vs_*.csv`: paired comparisons with all seeds. Screening tables are exploratory.',
        '- `*_factor_contrasts.csv`: explicit coverage, data-size, component and matched-estimator changes.',
        '- `system_holdout_gains.csv`: ensemble audit results, not official Kaggle scores.',
        '- `submission_*.csv`: all test candidates scored, using template-sized ranked outputs.',
        '', '## Recovery', '',
        'Only ZIPs successfully downloaded to your computer survive deletion of local Colab storage.',
        'Browser download requests can be blocked or missed during disconnection. No Google Drive backup is enabled.',
        'Resume a trusted v3 ZIP with unchanged scientific settings and package versions. V2 ZIPs are not v3 checkpoints.',
        'Completed jobs are reused; self-contained GNN epoch checkpoints retain optimizer, RNG and best-model state.',
        'A session-budget stop is incomplete work. Re-run the same notebook; it does not require new experiment choices.',
        '', '## Limits', '',
        'Larger training sets change both information and optimizer steps under a fixed epoch limit.',
        'The configurable-node seeds are nested by budget; full retained graphs need not be strict subsets.',
        'Intervals resample whole graphs and are exploratory, without multiple-comparison correction.',
        'The two folds are training variations, not independent copies of the holdout.',
        'No guarantee of an OOM-free run or of reaching top 10%.', '']
    (output/'RUN_SUMMARY.md').write_text('\n'.join(lines))


def run_study(root,output,scratch,engine_path,cfg,on_checkpoint=None):
    validate_config(cfg)
    if not cfg['folds'] or len(set(cfg['folds']))!=len(cfg['folds']):raise ValueError('Distinct folds required')
    if max(cfg['data_config_counts']+[cfg['large_fit_configs']])>cfg['train_pool_configs']:raise ValueError('Training pool is too small')
    if 24 not in cfg['data_graph_counts'] or 64 not in cfg['data_config_counts']:raise ValueError('Keep the 24 graph / 64 configuration control')
    root,output,scratch=map(lambda p:Path(p).resolve(),[root,output,scratch]);engine_path=Path(engine_path).resolve()
    if any(a==b or a in b.parents or b in a.parents for a,b in [(root,scratch),(output,scratch),(root,output)]):
        raise ValueError('Data, scratch and output directories must be separate')
    output.mkdir(parents=True,exist_ok=True);scratch.mkdir(parents=True,exist_ok=True)
    marker=scratch/'.v3_owned'
    if not marker.exists() and any(scratch.iterdir()):raise ValueError('Refusing nonempty unmarked scratch directory')
    marker.touch();versions={n:importlib.metadata.version(n) for n in ['numpy','pandas','scipy','scikit-learn','torch','joblib','psutil']}
    sources={p.name:digest(p.read_text()) for p in engine_path.parent.glob('study_*.py')}
    signature=digest(dict(cfg=scientific_config(cfg),versions=versions,sources=sources))
    identity=output/'run_identity.json'
    if identity.exists() and json.loads(identity.read_text())['signature']!=signature:
        raise ValueError('Code/settings/packages changed, or this is a v2 checkpoint. Use the matching v3 code/settings.')
    write_json(identity,dict(version=VERSION,signature=signature,versions=versions,config=cfg,sources=sources))
    for path in engine_path.parent.glob('study_*.py'):
        target=output/path.name
        if target.exists() and target.read_bytes()!=path.read_bytes():raise ValueError('Archived code differs')
        if not target.exists():shutil.copy2(path,target)
    deadline=time.monotonic()+cfg['session_hours']*3600;last_checkpoint=time.monotonic()
    states={c:'pending' for c in cfg['collections']};active=None

    def checkpoint(force=False):
        nonlocal last_checkpoint
        if not force and time.monotonic()-last_checkpoint<cfg['checkpoint_minutes']*60:return
        write_summary(output,cfg,states)
        archive=make_bundle(output,cfg['disk_reserve_gib'])
        write_json(output/'last_checkpoint.json',dict(created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),archive=archive.name,
                   note='ZIP created; browser/computer receipt cannot be verified by the notebook'))
        if on_checkpoint:on_checkpoint(archive)
        last_checkpoint=time.monotonic()
        print(f'Checkpoint ZIP: {archive}',flush=True)

    def worker(job,status,limits=None):
        if folder_bytes(output)>cfg['output_limit_gib']*GIB:raise OSError('Output budget reached; preserve/download results')
        result=execute(job,engine_path,limits or cfg,deadline,status,heartbeat=checkpoint)
        checkpoint();return result

    def cleanup():
        work=scratch/'work'
        if marker.exists() and work.exists():shutil.rmtree(work)

    def prepare_for(collection,directory,fold,exp):
        effective=effective_config(cfg,exp,collection);m=manifest(root,collection,cfg,fold)
        manifest_path=directory/'manifests'/f'fold_{fold}.json'
        if manifest_path.exists() and json.loads(manifest_path.read_text())!=m:raise ValueError('Source manifest changed')
        write_json(manifest_path,m)
        geometry=exp.get('profile') if exp['model']=='gnn' else None
        key=digest(dict(manifest=m,geometry=geometry,version=VERSION))[:16]
        work=scratch/'work';cache=work/'cache_identity.json'
        if cache.exists() and json.loads(cache.read_text())['key']==key and (work/'ready.json').exists():return effective,work,True
        cleanup();work.mkdir(exist_ok=True)
        prep=directory/'preparations'/f'fold_{fold}_{key}';prep.mkdir(parents=True,exist_ok=True)
        status=prep/'status.json';status.unlink(missing_ok=True)
        ok=worker(dict(action='prepare',cfg=effective,root=str(root),manifest=m,scratch=str(work),output=str(prep)),status)
        if ok:write_json(cache,dict(key=key))
        return effective,work,ok

    def run_jobs(collection,directory,catalog,seeds,action='trial'):
        ok_all=True
        jobs=[(fold,e,seed) for fold in cfg['folds'] for e in catalog for seed in seeds]
        jobs.sort(key=lambda x:(x[0],json.dumps(x[1].get('profile') if x[1]['model']=='gnn' else None,sort_keys=True),x[1]['name'],x[2]))
        for fold,exp,seed in jobs:
            path=trial_dir(directory,fold,exp['name'],seed);path.mkdir(parents=True,exist_ok=True)
            marker_path=path/('status.json' if action=='trial' else 'holdout_status.json')
            if marker_path.exists() and json.loads(marker_path.read_text())['status']=='ok':continue
            if time.monotonic()>=deadline:return False
            effective,work,ready=prepare_for(collection,directory,fold,exp)
            if not ready:ok_all=False;continue
            print(f'{collection} | {exp["stage"]} | fold {fold} | {exp["name"]} | seed {seed} | {action}',flush=True)
            ok=worker(dict(action=action,cfg=effective,experiment=exp,seed=seed,collection=collection,
                           scratch=str(work),output=str(path)),marker_path)
            ok_all=ok_all and bool(ok)
            if ok is None:return False
        return ok_all

    try:
        # Verify every fold/source before allowing completed-job shortcuts.
        for collection in cfg['collections']:
            directory=output/collection.replace(':','_');directory.mkdir(exist_ok=True)
            for fold in cfg['folds']:
                m=manifest(root,collection,cfg,fold);p=directory/'manifests'/f'fold_{fold}.json'
                if p.exists() and json.loads(p.read_text())!=m:raise ValueError('Source manifest changed; do not mix data versions')
                write_json(p,m)
        for collection in cfg['collections']:
            active=collection;directory=output/collection.replace(':','_')
            if (directory/'complete.json').exists():states[collection]='complete (resumed)';continue
            if time.monotonic()>=deadline:states[collection]='incomplete: session budget';continue
            catalog=tabular_plan(collection,cfg);coverage=None
            if collection in cfg['gnn_collections']:
                states[collection]='profiling graph budgets'
                # Probe the largest training graphs by three independent static dimensions, with no runtime labels.
                m=json.loads((directory/'manifests'/f'fold_{cfg["folds"][0]}.json').read_text())
                train=[root/r['relative_path'] for r in m['records'] if r['split']=='train']
                stats=[(p,metadata(p,'node_feat')[0][0],metadata(p,'node_config_feat')[0][1],metadata(p,'edge_index')[0][0]) for p in train]
                probe_paths=list(dict.fromkeys(str(max(stats,key=lambda x:x[k])[0]) for k in [1,2,3]))[:cfg['probe_graphs']]
                eligible=[]
                for nodes in cfg['graph_budgets']:
                    exp=next(e for e in coverage_plan(cfg) if e['name']==f'G_full_n{nodes}')
                    effective=effective_config(cfg,exp,collection);destination=directory/'probes'/str(nodes);destination.mkdir(parents=True,exist_ok=True)
                    result=worker(dict(action='probe',cfg=effective,paths=probe_paths,output=str(destination)),destination/'status.json',
                                  dict(cfg,worker_ram_gib=cfg['probe_ram_gib']))
                    if result:eligible.append(nodes)
                    if result is None:break
                write_json(directory/'resource_eligibility.json',dict(eligible_nodes=eligible,attempted_nodes=cfg['graph_budgets'],
                    note='Budgets are excluded on pilot resource failure, not on held-out accuracy'))
                if len(eligible)<1 or cfg['graph_budgets'][0] not in eligible:
                    states[collection]='incomplete: reference probe failed or budget ended';continue
                first=[e for e in coverage_plan(cfg) if e['profile']['nodes'] in eligible]
                catalog += first;write_json(directory/'experiment_plan.json',catalog)
                states[collection]='screening graph coverage on all development folds'
                run_jobs(collection,directory,first,[cfg['seeds'][0]])
                if time.monotonic()>=deadline:states[collection]='incomplete: coverage screening';continue
                if not all(finished(directory,e['name'],cfg,[cfg['seeds'][0]]) for e in first):
                    states[collection]='incomplete: coverage screening failures';continue
                coverage=select_coverage(directory,first,cfg,collection)
                catalog += followup_plan(cfg,coverage['profile'])
            write_json(directory/'experiment_plan.json',catalog)
            remaining=[e for e in catalog if not finished(directory,e['name'],cfg,[cfg['seeds'][0]])]
            states[collection]='screening training-data grid, controls and tabular candidates'
            run_jobs(collection,directory,remaining,[cfg['seeds'][0]])
            if not all(finished(directory,e['name'],cfg,[cfg['seeds'][0]]) for e in catalog):
                states[collection]='incomplete: screening jobs (inspect logs)';continue
            plan=confirmation_plan(directory,catalog,cfg,collection,coverage)
            confirmed=[e for e in catalog if e['name'] in plan['names']]
            states[collection]='three-seed confirmation'
            run_jobs(collection,directory,confirmed,cfg['seeds'])
            if not all(finished(directory,e['name'],cfg) for e in confirmed):
                states[collection]='incomplete: confirmation';continue
            lock=lock_selection(directory,catalog,cfg,collection,plan['names'])
            print(f'LOCKED on development folds: {lock["components"]}',flush=True)
            states[collection]='auditing confirmed comparisons'
            run_jobs(collection,directory,confirmed,cfg['seeds'],'holdout')
            if all(finished(directory,e['name'],cfg,action='holdout') for e in confirmed):
                write_json(directory/'complete.json',dict(status='complete'));states[collection]='complete'
            else:states[collection]='incomplete: audit'
            summarize(directory,cfg,collection);cleanup();checkpoint(force=True)
        if cfg['generate_submissions'] and all((output/c.replace(':','_')/'complete.json').exists() for c in cfg['collections']):
            from study_submission import export_submissions
            export_submissions(root,output,engine_path,cfg,deadline,
                               lambda job,engine,limits,until,status:worker(job,status))
        elif cfg['generate_submissions']:
            write_json(output/'submission_status.json',dict(status='pending',reason='Required stages incomplete'))
    except KeyboardInterrupt:
        if active:states[active]='incomplete: interrupted'
        print('Interrupted: creating a recovery archive.',flush=True)
    except Exception as exc:
        if active:states[active]=f'incomplete: {type(exc).__name__}: {exc}'
        write_json(output/'controller_error.json',dict(type=type(exc).__name__,error=str(exc)))
        raise
    finally:
        for collection in cfg['collections']:
            directory=output/collection.replace(':','_')
            if directory.exists():
                try:summarize(directory,cfg,collection)
                except Exception as exc:write_json(directory/'summary_error.json',dict(error=str(exc)))
        cleanup();write_summary(output,cfg,states);checkpoint(force=True);release_memory()
    return output.parent/(output.name+'_results.zip')
'''
(RUNTIME_DIR / "study_controller.py").write_text(SOURCE)
del SOURCE

In [ ]:
import importlib, json, shutil, zipfile
import numpy as np
import pandas as pd
import psutil
sys.path.insert(0,str(RUNTIME_DIR))
import study_engine, study_gnn, study_common, study_plan, study_submission, study_controller
for module in [study_engine,study_gnn,study_common,study_plan,study_submission,study_controller]:
    importlib.reload(module)
CONFIG = study_plan.default_config()
CONFIG.update(session_hours=SESSION_HOURS,checkpoint_minutes=CHECKPOINT_MINUTES,
              worker_ram_gib=WORKER_RAM_GIB,system_reserve_gib=SYSTEM_RESERVE_GIB)
print(f"Available RAM: {psutil.virtual_memory().available/1024**3:.1f} GiB")
print(f"Available disk: {shutil.disk_usage(BASE_DIR).free/1024**3:.1f} GiB")
print("Local output:", OUTPUT_DIR)
print("Automatic ZIP downloads:", AUTO_DOWNLOAD_CHECKPOINTS)
if not AUTO_DOWNLOAD_CHECKPOINTS:
    print("WARNING: local-only progress will be lost if the VM is deleted; download ZIPs manually.")

## 3. Find the competition data and check the test template

Download requires accepting the competition rules on Kaggle. Credentials can be stored as
`KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets; they are not printed or archived.
Original competition files remain on disk. The notebook creates no expanded raw configuration
cache. Prefer at least 20 GiB free after extracting the source data, with more space for a
large run and simultaneous old/new ZIPs. Disk guards do not guarantee against quota changes.

In [ ]:
def locate_data():
    bases=([Path(DATA_ROOT)] if DATA_ROOT else [])+[BASE_DIR/"predict-ai-model-runtime",BASE_DIR/"data",BASE_DIR]
    for base in bases:
        for path in [base,base/"npz_all/npz",base/"npz"]:
            if (path/"layout").is_dir() and (path/"tile").is_dir():return path.resolve()
    return None
NPZ_ROOT=locate_data()
if NPZ_ROOT is None and DOWNLOAD_IF_MISSING:
    from importlib.metadata import version,PackageNotFoundError
    try:installed=tuple(int(x) for x in version("kagglehub").split(".")[:3])
    except (PackageNotFoundError,ValueError):installed=(0,0,0)
    if installed<(0,4,3):
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-cache-dir","kagglehub>=0.4.3,<2"])
    try:
        from google.colab import userdata
        for name in ["KAGGLE_USERNAME","KAGGLE_KEY"]:
            if not os.environ.get(name):
                try:
                    value=userdata.get(name)
                    if value:os.environ[name]=value
                except Exception:pass
    except ImportError:pass
    if shutil.disk_usage(BASE_DIR).free<25*1024**3:
        raise OSError("Need more free disk before downloading data and saving experiments.")
    import kagglehub
    kagglehub.competition_download("predict-ai-model-runtime",output_dir=str(BASE_DIR/"predict-ai-model-runtime"))
    NPZ_ROOT=locate_data()
if NPZ_ROOT is None:raise FileNotFoundError("Set DATA_ROOT to the extracted competition data.")
CONFIG["sample_submission"]=str(study_submission.locate_template(NPZ_ROOT,SAMPLE_SUBMISSION))
print("Data:", NPZ_ROOT)
print("Test/template preflight:", study_submission.preflight(NPZ_ROOT,CONFIG))

## 4. Restore a checkpoint and inspect the automatic plan

Leave the restore path empty on the first run. After a VM reset, upload a **v3** ZIP and run all
cells with its path. Restoring requires an empty output directory. If the current runtime still
has your results, leave `RESTORE_ZIP` empty and rerun section 6 to continue directly.

The two development folds use different held-out training graphs. A model only scores its
own development fold; we never validate it on another fold's graphs that it trained on.
Official valid has already been inspected during v2, so v3 calls it an **audit**, not an
untouched final test. Architecture families can still cross graph splits.

In [ ]:
if RESTORE_ZIP:
    with zipfile.ZipFile(RESTORE_ZIP) as archive:
        identity=json.loads(archive.read("run_identity.json"))
        if identity.get("version")!="3.0.0":raise ValueError("Use a v3 checkpoint; v2 is a separate study.")
    study_controller.restore_archive(RESTORE_ZIP,OUTPUT_DIR,CONFIG["output_limit_gib"])
    print("Restored:",RESTORE_ZIP)
print("Folds:",CONFIG["folds"],"| confirmation seeds:",CONFIG["seeds"])
print("Graph node budgets:",CONFIG["graph_budgets"])
print("GNN training graph counts:",CONFIG["data_graph_counts"],"(0 means all available training graphs)")
print("Configurations per training graph:",CONFIG["data_config_counts"])
print("Checkpoint download interval:",CONFIG["checkpoint_minutes"],"minutes")
plan=study_plan.coverage_plan(CONFIG)+study_plan.followup_plan(CONFIG,study_plan.profile(2048))
display(pd.DataFrame(plan)[["stage","name","model","train_graphs","configs","profile"]])
print("The table illustrates follow-up experiments at 2,048 nodes; the actual budget is selected automatically.")

## 5. What runs, and why

| Stage | Experiment | What stays fixed |
|---|---|---|
| Capacity pilots | Real large graphs, full training batch, forward/backward; no runtime labels | Resource thresholds; batch size 16 |
| Graph coverage | G0 and G_full at 512/1,024/2,048/4,096 nodes | 24 training graphs × 64 configurations, architecture, loss and evaluation candidates |
| Component controls | Masks, context and no message passing at the selected budget | Graph budget and training data |
| Sampler control | Connected configurable seeds versus uniform configurable seeds | Caps, model recipe and training-data IDs |
| Training-data grid | 24 / 40 / all training graphs × 64 / 128 / 256 configurations | Selected graph budget and full GNN recipe |
| Matched HGB | Same graph/configuration IDs for each training-data recipe | Full tabular representation and HGB recipe |
| Tabular development | Basic/full features, more graphs, more configurations, both, deeper HGB, Extra Trees | Explicit per-experiment settings |
| Confirmation | Key controls, the data-grid corners and development leaders repeated across 3 seeds | Same 2 graph folds |
| Selection and audit | Freeze a development-selected model/blend, then audit confirmed methods | No audit-based model selection |
| Test export | Baseline, tabular-only and selected system CSVs | Frozen fitted models; no audit retraining |

All candidates are screened with seed 42 on **both folds**. Key comparisons and selected leaders
then receive seeds 137 and 2026 too. Screening and confirmed results are saved separately.
The 2×2 training-data corners isolate more graphs, more configurations and their combination;
intermediate grid points remain available for diagnosis and can qualify as finalists.

A graph budget must pass a pilot before training. Exclusions remain explicit. The same physical
configuration batch is retained, so the pairwise loss is not silently changed by microbatching.
Sampling uses a fixed configurable-node permutation to make its seed prefixes nested. The full
retained graphs need not be strict subsets. The 512-node control is a v3 reference under this
sampling implementation, not a reconstruction of v2's exact sampled nodes.

More data also means more optimizer updates under a fixed epoch cap. This is a practical
training-data study, not a compute-matched scaling-law experiment. Early stopping uses a
fixed development subset; candidate selection uses the complete sampled development set.

Selected blends must improve the mean development score and avoid a drop larger than 0.02
on either fold relative to B0. Baseline-only is always a candidate. This is a development rule,
not a guarantee that every audit or leaderboard slice improves.

## 6. Run everything and download checkpoints automatically

Allow this browser page to download multiple files. Check your computer's Downloads folder
for successive copies of `sc4000_gnn_v3_results.zip`. Your browser may append `(1)`, `(2)`, etc.
The most recent successfully downloaded copy is the recovery point.

A ZIP is created and its download requested every 30 minutes, after each collection and at
normal stop/interruption. It includes completed trials and self-contained active GNN epoch
checkpoints. It excludes competition data and temporary feature matrices. On a runtime reset,
work after the last downloaded checkpoint can be lost. No keep-alive or runtime-limit bypass
is used. The notebook cannot confirm that your browser finished saving a file.

In [ ]:
def download_checkpoint(path):
    import datetime
    print("ZIP ready at",datetime.datetime.now(datetime.timezone.utc).isoformat(),":",path)
    print("Check that the download reaches your computer. Colab local storage is temporary.")
    if AUTO_DOWNLOAD_CHECKPOINTS:
        try:
            from google.colab import files
            files.download(str(path))
        except ImportError:
            print("Outside Colab: copy this ZIP from the displayed path.")
        except Exception as exc:
            print("Browser download request failed:",type(exc).__name__,str(exc))
            print("Download the ZIP manually from the Colab Files sidebar.")

RESULTS_ZIP=study_controller.run_study(NPZ_ROOT,OUTPUT_DIR,SCRATCH_DIR,
                                     RUNTIME_DIR/"study_engine.py",CONFIG,
                                     on_checkpoint=download_checkpoint)

## 7. Review completeness and results

Graph coverage and training-data comparisons are separate in the saved plan. Use confirmed
comparisons for report claims and retain negative results. A single-seed screening gain is not
three-seed evidence. Layout uses sampled Kendall tau-b; tile uses sampled top-five regret.
These local metrics are not Kaggle leaderboard scores. Bootstrap intervals are exploratory.
`*_factor_contrasts.csv` identifies the factor changed and its paired gain, including matched
GNN/HGB comparisons. Relative percentages are omitted when the reference score is too low
or negative to make them useful. Screening and confirmed tables remain separate.

In [ ]:
import matplotlib.pyplot as plt
state=json.loads((OUTPUT_DIR/"run_state.json").read_text())
display(pd.DataFrame(state["collections"].items(),columns=["collection","status"]))
for collection in CONFIG["collections"]:
    directory=OUTPUT_DIR/collection.replace(":","_")
    print("\n",collection)
    lock=directory/"selection_lock.json"
    if lock.exists():print("Selected:",json.loads(lock.read_text())["components"])
    path=directory/"system_holdout_gains.csv"
    if path.exists():display(pd.read_csv(path))
    path=directory/"holdout_confirmed_factor_contrasts.csv"
    if path.exists():display(pd.read_csv(path))
    path=directory/"all_trials.csv"
    if path.exists():
        trials=pd.read_csv(path)
        print("Completed fits:",len(trials),"| observed maximum worker GiB:",round(trials.peak_rss_gib.max(),3))
        screen=trials[trials.seed==CONFIG["seeds"][0]]
        plot=screen[screen.experiment.str.startswith("G_full_n")].groupby("experiment").dev_score.mean()
        if len(plot):
            fig,ax=plt.subplots(figsize=(7,3));plot.plot.bar(ax=ax)
            ax.set_ylabel("Mean development tau across folds — screening seed")
            ax.set_title(collection+": graph-budget screening");fig.tight_layout()
            fig.savefig(directory/"coverage_screening.png",dpi=150);plt.show();plt.close(fig)
submission_status=OUTPUT_DIR/"submission_status.json"
if submission_status.exists():print("Submission status:",json.loads(submission_status.read_text()))
for path in OUTPUT_DIR.glob("submission_*.csv"):print("Candidate CSV:",path)
RESULTS_ZIP=study_controller.make_bundle(OUTPUT_DIR,CONFIG["disk_reserve_gib"])
download_checkpoint(RESULTS_ZIP)

## 8. What to send back

Send the most recent **results ZIP**, including partial results if the run stopped. It contains
code/settings, graph/configuration manifests, resource pilots, selection decisions, training
histories, models, scores, prediction IDs, failures and submission CSVs when complete.
You do not need to select the next experiment manually. Resume until the required stages
and submission export are complete, or share partial evidence if a resource failure blocks work.

`submission_baseline.csv`, `submission_tabular.csv` and `submission_selected.csv` are candidate
files. The notebook does not upload to Kaggle. All models use only their fold's training graphs;
the final test ensemble averages fold and seed models. It does not retrain on the audit labels.

## Attribution

This is an independent extension of the project's v2 implementation. Established methods are
not new inventions. The prior GNN already had message passing, numeric node features, opcode
embeddings and pairwise loss. The research question here is whether retaining more graph
context and using more labelled training examples improves the controlled references.

- [GraphSAGE paper](https://arxiv.org/abs/1706.02216): neighbourhood sampling and aggregation.
- [1st-place authors](https://github.com/thanhhau097/google_fast_or_slow): representation/preprocessing inspiration.
- [2nd-place authors](https://github.com/dmitrii-khizbullin/kaggle_latenciaga): ranking and configuration comparisons.
- [3rd-place authors](https://github.com/jafluri/kaggle_tpu_graph): representation and seed variability.
- [5th-place write-up](https://www.kaggle.com/competitions/predict-ai-model-runtime/discussion/456093): dimension semantics.
- [Official competition data](https://www.kaggle.com/competitions/predict-ai-model-runtime/data): NPZ schema and normalization.
- [PyTorch checkpoint guide](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html): training-state recovery.
- [Colab FAQ](https://research.google.com/colaboratory/faq.html): temporary runtimes and resource availability.

No competitor solution code, weights or predictions are embedded. Cite inspiration and distinguish
new project experiments from global method novelty. The notebook does not guarantee top 10%.